# ⏱️ 05 · Asymptotics: how fast functions grow
### Comparing how functions grow: Big-O as a set of functions and its quantifier definition, witnesses, the rules for sums, products and polynomials, logarithms ≪ polynomials ≪ exponentials, Ω, Θ, o and ω — and why growth rates decide which bioinformatics computations are feasible

**BiologyToMath** · Part B — Foundations: logic, sets, functions, sequences and graphs · Module 05 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/05_asymptotics_growth_of_functions.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SymPy (limits and exact algebra), SciPy (`optimize.brentq` to find crossing points), Matplotlib, `time` (to measure running times) |
| **Prerequisites** | [Module 02 · 🔣 Logic: the grammar of precise reasoning](02_logic.ipynb) (∀, ∃, negating quantifiers), [Module 03 · 🧺 Sets and functions: collections and mappings](03_sets_and_functions.ipynb) (sets, functions, domain and codomain) and [Module 04 · 🔤 Tuples, sequences and strings: when order matters](04_tuples_sequences_strings.ipynb) (counting tuples: $4^k$, $2^n$). High-school algebra: powers and logarithms (refreshed here); no calculus needed — limits are introduced informally. |
| **Estimated time** | 110 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Asymptotics: comparing the growth of functions $f, g: \mathbb R^{\ge 0}\to\mathbb R^{\ge 0}$ ("does f grow no faster than g?"); Big-O as a set of functions, $O(g) = \{h \mid \exists x_0\ge 0, c>0: \forall x\ge x_0: h(x)\le c\,g(x)\}$, ≥ vs >, and the pedantry of $f\in O(g)$ vs $f = O(g)$; the example 4x vs x² and its witnesses; the four properties (sum ↔ max, transitivity, sums, products); polynomials $p(x)\in O(x^k)$ and the idea of the proof; log ≪ polynomial ≪ exponential, $\log x\in O(x^\epsilon)$, the base of the log does not matter, $p(x)\in O(b^x)$ for $b>1$, $b^x\in O(c^x)\iff b\le c$; Ω, Θ, o and ω, $f\in O(g)\iff g\in\Omega(f)$; the worked examples ($3x^2+2\in O(x^2)$? $x^3\in O(x^2)$? $3x+4y+5xy$ in O(x), O(y), O(xy)? the even/odd function that is $x^3$ or $x$); continuous vs discrete exponential growth (if α ≈ β, which grows faster?) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. Running times are measured on the machine that runs the notebook, so your numbers will differ from the ones quoted in the text (the growth *rates* will not). DNA and the sequences compared in Section 8 are **SIMULATED** (`simulate_dna`, random arrays); genome sizes, doubling times and the bacterial cell mass are rounded, illustrative values.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Read** the quantifier definition of $O(g)$ — ∃ $c$, ∃ $x_0$, ∀ $x \ge x_0$ — and **explain** why "$f\in O(g)$" is more precise than "$f = O(g)$".
2. **Find** witnesses $(c, x_0)$ that prove $f\in O(g)$ (e.g. $4x\in O(x^2)$, $3x^2+2\in O(x^2)$) and **disprove** membership by negating the definition ($x^2\notin O(4x)$, $x^3\notin O(x^2)$).
3. **Prove** and **apply** the rules of Big-O — sum ↔ max, transitivity, sums, products — and **show** that every polynomial of degree $k$ lies in $O(x^k)$.
4. **Order** the standard growth rates (logarithms ≪ polynomials ≪ exponentials ≪ factorials) and **explain** why the base of a logarithm does not matter while the base of an exponential does: $b^x\in O(c^x)\iff b\le c$.
5. **Define** Ω, Θ, o and ω, **use** the duality $f\in O(g)\iff g\in\Omega(f)$, and **classify** pairs of functions with limits of ratios — and recognise when that test fails.
6. **Solve** the worked exercises: $3x+4y+5xy$ in two variables, and the even/odd function that cannot be compared with $x^2$ at all.
7. **Measure** the running times of bioinformatics algorithms (all-pairs comparison, Needleman–Wunsch alignment, exhaustive motif search) and **read** their growth rates off log–log plots.
8. **Explain** with asymptotics why phylogenetic tree search needs heuristics — there are $(2n-3)!!$ rooted trees — and why exponential population growth cannot last.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the "zoo" of growth functions | 5 min |
| 1 · Comparing growth | 4x vs x²; what happens when the input doubles | 10 min |
| 2 · Big-O as a set of functions | the quantifier definition; witnesses; disproving membership | 15 min |
| 3 · Rules for Big-O | sum ↔ max, transitivity, sums, products — with proofs | 10 min |
| 4 · Polynomials | $p(x)\in O(x^k)$; $3x^2+2$ vs $x^2$; $x^3\notin O(x^2)$; limits of ratios | 10 min |
| 5 · Logs, polynomials, exponentials | the growth hierarchy; bases; crossing points; "how big can n be?" | 15 min |
| 6 · Ω, Θ, o, ω | the whole family, duality, limit test and its limits | 15 min |
| 7 · Worked exercises | $3x+4y+5xy$; the even/odd function; continuous vs discrete growth | 10 min |
| 8 · Asymptotics in bioinformatics | all pairs, alignment, motif search, tree counts, population growth | 15 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

A sequencing core hands you 10 million reads. Your supervisor suggests four analyses:

1. compare **every read with every other read** to find duplicates;
2. **align** every read (150 bases) against the 3.1-billion-base human genome with the textbook dynamic-programming
   algorithm;
3. search the reads for **every possible 15-letter motif**;
4. find the best **phylogenetic tree** relating 50 bacterial strains by checking every possible tree.

Which of these will finish before you graduate? Nobody can answer by running them — the point is to know *before*
running. The answer depends not on how fast your computer is but on **how fast the amount of work grows** with the
size of the input: like $n^2$ (all pairs), like $n\cdot m$ (alignment), like $4^k\cdot L$ (motifs), or like
$(2n-3)!! = 1\cdot 3\cdot 5\cdots(2n-3)$ (trees). **Asymptotics** is the mathematics that compares such growth rates
precisely, with the quantifiers of Module 02 and the sets of Module 03.

| Biological / computational question | Mathematical tool | Where in this module |
|---|---|---|
| Does doubling the number of samples double, quadruple or square the work? | growth of $x$, $x^2$, $2^x$ | Section 1 |
| Is "this algorithm is $O(n^2)$" a precise claim? What exactly does it promise? | Big-O as a set; witnesses | Section 2 |
| A pipeline runs QC, sorting and all-pairs comparison — which step dominates? | sum ↔ max, sums, products | Sections 3–4 |
| Why are "exponential" algorithms hopeless and "logarithmic" ones almost free? | log ≪ poly ≪ exp; $b^x\in O(c^x)\iff b\le c$ | Section 5 |
| Are two algorithms equally fast "up to constants"? | Θ, o, ω, limits of ratios | Section 6 |
| Alignment of reads of fixed length: $O(nm)$ or $O(n)$? | two-variable Big-O | Section 7 |
| How many trees for 50 species? How long can bacteria grow exponentially? | super-exponential growth; logistic vs exponential | Section 8 |

> 🧪 **Simulated data, with a known truth.** The sequences we align and compare are **SIMULATED** by code
> (`simulate_dna` and random integer arrays), so their lengths and numbers are exactly what we choose — which is all
> that matters when measuring how running time grows. Running times are real measurements on your machine.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import itertools
import math
import time

import sympy as sp
from scipy.optimize import brentq

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). They simulate random DNA and count k-mers; Section 8 times them. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
from collections import Counter

import numpy as np


NUCLEOTIDES = ("A", "C", "G", "T")


def simulate_dna(length: int, transitions=None, initial=None, seed: int = 0) -> str:
    """Random DNA string.  Without ``transitions`` the bases are i.i.d. uniform; otherwise they
    follow a first-order Markov chain with the given 4 × 4 transition matrix (order A, C, G, T)
    and initial distribution ``initial`` (default uniform)."""
    rng = np.random.default_rng(seed)
    initial = np.full(4, 0.25) if initial is None else np.asarray(initial, dtype=float)
    if transitions is None:
        return "".join(rng.choice(list(NUCLEOTIDES), size=length, p=initial))
    P = np.asarray(transitions, dtype=float)
    states = [rng.choice(4, p=initial)]
    for _ in range(length - 1):
        states.append(rng.choice(4, p=P[states[-1]]))
    return "".join(NUCLEOTIDES[s] for s in states)


def kmer_counts(seq: str, k: int) -> dict:
    """Counts of every overlapping k-mer (substring of length k) of ``seq``, sorted alphabetically."""
    counts = Counter(seq[i:i + k] for i in range(len(seq) - k + 1))
    return dict(sorted(counts.items()))

print("🧰 toolkit loaded:", ", ".join(sorted(['NUCLEOTIDES', 'kmer_counts', 'simulate_dna'])))

In [ ]:
# ── 0.1 The "zoo" of growth functions used throughout this module ──
GROWTH = {                                      # name → function of x (NumPy, works on arrays)
    "log₂ x": lambda x: np.log2(x),
    "√x": lambda x: np.sqrt(x),
    "x": lambda x: x,
    "x log₂ x": lambda x: x * np.log2(x),
    "x²": lambda x: x ** 2,
    "x³": lambda x: x ** 3,
    "2^x": lambda x: 2.0 ** x,
}
xs = np.array([2, 4, 8, 16, 32, 64])
table = pd.DataFrame({name: f(xs.astype(float)) for name, f in GROWTH.items()}, index=xs)
table.index.name = "x"
print(table.to_string(float_format=lambda v: f"{v:,.0f}" if v >= 10 else f"{v:.2f}"))

🔍 **What just happened?** Each column is one function evaluated at $x = 2, 4, 8, \dots, 64$ (each row doubles $x$).
Read down the columns: $\log_2 x$ creeps up by exactly 1 per row (from 1 to 6); $\sqrt x$ grows by a factor
$\sqrt 2\approx 1.41$ per row; $x$ doubles; $x^2$ quadruples (4 → 4,096); $x^3$ is multiplied by 8 (8 → 262,144); and
$2^x$ *squares* — from 4 to about $1.8\times10^{19}$ at $x = 64$, a number with 20 digits. At $x = 2$ all of them are
small and similar; the differences only appear as $x$ grows. That "eventually" is the whole subject of this module.

---
## 1 · Comparing growth: does $f$ grow no faster than $g$?

> 📍 **Where we are.** Module 04 produced numbers that explode — $4^k$ k-mers, $3^J$ genotype tuples, $2^n - 1$ binary
> numbers. To reason about such numbers (and about the running time of the programs that handle them) we want to
> compare two functions $f(x)$ and $g(x)$ by asking one question:

> **Does $f(x)$ grow no faster than $g(x)$?**

Throughout, the functions are $f, g: \mathbb R^{\ge 0}\to\mathbb R^{\ge 0}$: they take a non-negative input (a number of
sequences, a sequence length, a number of species) and return a non-negative output (a running time, an amount of
memory, a number of objects). Two features of the question are deliberate:

* **Constants should not matter.** A computer that is 10 times faster turns running time $f$ into $f/10$; the *growth*
  of the work is unchanged.
* **Only large inputs matter.** Every method is fast on 5 sequences; the question is what happens for 5 million.

🧮 **Math refresher — powers and logarithms.** $b^x$ means $b$ multiplied by itself $x$ times (for whole $x$), with the
rules $b^{x+y} = b^x b^y$ and $(b^x)^y = b^{xy}$. The **logarithm** undoes the power: $\log_b y$ is the exponent that
turns $b$ into $y$, so $\log_b(b^x) = x$ — for example $\log_2 1024 = 10$ because $2^{10} = 1024$, and $\log_{10}1000 = 3$.
Logarithms turn products into sums, $\log(ab) = \log a + \log b$, and powers into products, $\log(a^k) = k\log a$.
The **natural logarithm** $\ln = \log_e$ uses the base $e\approx 2.718$; in this course "log" without a base means ln.
Any two logarithms differ by a constant factor: $\log_b x = \dfrac{\ln x}{\ln b}$ (Section 5 uses this).

**The first example.** Let $f(x) = 4x$ and $g(x) = x^2$. For small $x$ the line $4x$ is above the parabola; for large
$x$ the parabola wins.

In [ ]:
# ── 1.1 f(x) = 4x versus g(x) = x²: who is larger, and what does the ratio do? ──
x = np.linspace(0, 8, 401)
f_lin, g_sq = 4 * x, x ** 2
for x_val in [1, 2, 3, 4, 5, 6, 8]:
    print(f"x = {x_val}: 4x = {4 * x_val:3d}, x² = {x_val ** 2:3d} → {'4x larger' if 4 * x_val > x_val ** 2 else 'equal' if 4 * x_val == x_val ** 2 else 'x² larger'}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(x, f_lin, color="tab:blue", label="f(x) = 4x")
axes[0].plot(x, g_sq, color="tab:red", label="g(x) = x²")
axes[0].axvline(4, color="0.4", ls=":", label="x = 4")
axes[0].set_ylim(0, 60); axes[0].set_xlabel("x"); axes[0].set_ylabel("function value"); axes[0].legend()
axes[0].set_title("4x vs x²: the parabola wins from x = 4 on")
x_pos = x[1:]                                         # avoid dividing by zero at x = 0
axes[1].plot(x_pos, 4 * x_pos / x_pos ** 2, color="tab:purple")
axes[1].axhline(1, color="0.4", ls=":"); axes[1].axvline(4, color="0.4", ls=":")
axes[1].set_ylim(0, 6); axes[1].set_xlabel("x"); axes[1].set_ylabel("ratio f(x) / g(x) = 4 / x")
axes[1].set_title("the ratio f/g falls below 1 at x = 4 and tends to 0")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The table and the left panel agree: $4x > x^2$ for $0 < x < 4$, they are equal at $x = 4$
(both 16), and $x^2 > 4x$ for every $x > 4$. The right panel plots the **ratio** $f(x)/g(x) = 4x/x^2 = 4/x$, which drops
below 1 at $x = 4$ and keeps falling towards 0. Both views say "$4x$ grows no faster than $x^2$", and the ratio view
says more: $4x$ becomes a *negligible fraction* of $x^2$. Section 2 turns the first view into a definition; Section 6
uses the second.

### 1.2 What happens when the input doubles?

A practical way to feel a growth rate: double the input and look at the factor by which the work grows.

In [ ]:
# ── 1.2 Doubling the input: the factor f(2x) / f(x) for each function of the zoo ──
with np.errstate(over="ignore"):                        # 2^2000 overflows a float: NumPy returns inf
    for name, f in GROWTH.items():
        factors = [f(np.float64(2 * x_val)) / f(np.float64(x_val)) for x_val in (10, 100, 1000)]
        print(f"{name:9s}: f(2x)/f(x) at x = 10, 100, 1000 → " + ", ".join(f"{r:.4g}" for r in factors))

🔍 **What just happened?** Doubling the input multiplies $x$ by 2, $x^2$ by 4 and $x^3$ by 8, *whatever* $x$ is — for
a power $x^k$ the factor is always $2^k$, because $(2x)^k = 2^k x^k$. For $\log_2 x$ the factor shrinks towards 1
(1.30, 1.15, 1.10): a logarithmic algorithm barely notices a bigger input. $x\log_2 x$ grows by a bit more than 2.
For $2^x$ the factor is $2^x$ itself — about 1,024 at $x = 10$, $1.3\times10^{30}$ at $x = 100$ and an overflow
(`inf`) at $x = 1000$: doubling the input *squares* the work. If analysing 1,000 samples takes an hour with a
quadratic method, 2,000 samples take 4 hours; with an exponential method, adding even a few samples can be hopeless.

✏️ **Your turn.** Add `"x⁴": lambda x: x ** 4` and `"1.1^x": lambda x: 1.1 ** x` to `GROWTH` (cell 0.1) and re-run
1.2. Predict the doubling factor of $x^4$ before you run. Why does the factor for $1.1^x$ at $x = 10$ look harmless?

---
## 2 · Big-O as a set of functions

> 📍 **Where we are.** We want a precise version of "grows no faster than, ignoring constants and small inputs". The
> clever move of 02-680 is to define the **set** of all functions with that property, so that the question becomes a
> membership question — exactly the language of Module 03.

For functions $h, g: \mathbb R^{\ge 0}\to\mathbb R^{\ge 0}$,

$$O\big(g(x)\big) := \Big\{\, h(x) \;\Big|\; \exists x_0 \ge 0,\ c > 0 :\ \forall x \ge x_0 :\ h(x) \le c\cdot g(x) \Big\}.$$

*In words:* $O(g)$ ("big-O of $g$") is the set of all functions $h$ for which **there exist** a starting point $x_0$
and a constant $c$ such that, **for every** $x$ from $x_0$ on, $h(x)$ is at most $c$ times $g(x)$. The question "does
$f$ grow no faster than $g$?" becomes: **is $f(x)\in O(g(x))$?**

Read the definition one quantifier at a time (Module 02):

| Piece | Meaning | Why it is there |
|---|---|---|
| $\exists\, c > 0$ | some positive constant factor | constants do not matter: $c$ absorbs machine speed and units |
| $\exists\, x_0 \ge 0$ | some threshold | small inputs do not matter: only "eventually" counts |
| $\forall\, x \ge x_0$ | for **every** input beyond the threshold | not just at a few points — from $x_0$ on, always |
| $h(x)\le c\cdot g(x)$ | $h$ is below the scaled-up $g$ | "no faster than" |

Notice the explicit use of ≥ and > : $x_0\ge 0$ may be 0, but $c > 0$ must be strictly positive (with $c = 0$ the
inequality would force $h(x)\le 0$). A pair $(c, x_0)$ that makes the inequality true is called a pair of
**witnesses**: exhibiting one pair proves membership, because the definition only asks that *some* pair exists.

⚠️ **Pedantry detail.** You will often see $f(x) = O(g(x))$. This is not technically correct: it asks whether a function
*equals* a set of functions. The precise statement is $f(x)\in O(g(x))$. The "=" version is a widespread abuse of
notation — read it as "∈", and never "reverse" it: $x\in O(x^2)$ is true, but "$O(x^2) = x$" is meaningless.

In [ ]:
# ── 2.1 Checking candidate witnesses (c, x0) for 4x ∈ O(x²) on a fine grid ──
def check_witness(h, g, c: float, x0: float, x_max: float = 1000.0, n_points: int = 200_001) -> tuple:
    """Test h(x) ≤ c·g(x) on a grid of x ∈ [x0, x_max]; return (passes, first failing x or None).
    A grid can only DISPROVE a witness pair (by finding a failure) — it cannot prove ∀x ≥ x0."""
    grid = np.linspace(x0, x_max, n_points)
    bad = grid[h(grid) > c * g(grid) + 1e-12]
    return (len(bad) == 0, None if len(bad) == 0 else float(bad[0]))


h_lin = lambda x: 4 * x
g_sq = lambda x: x ** 2
for c, x0 in [(1, 4), (2, 2), (4, 1), (0.5, 8), (1, 3), (0.5, 4), (0.001, 100)]:
    ok, first_bad = check_witness(h_lin, g_sq, c, x0)
    verdict = "works ✓" if ok else f"fails ✗ (e.g. at x = {first_bad:.3f})"
    print(f"c = {c:<6} x0 = {x0:<4}: 4x ≤ c·x² for all grid points x ≥ x0? {verdict}")

🔍 **What just happened?** The pair $(c, x_0) = (1, 4)$ works: for $x\ge 4$, $4x\le x^2$ (divide both sides by $x>0$:
$4\le x$). But it is not the only pair: $(2, 2)$, $(4, 1)$ and $(0.5, 8)$ work too. In general, $4x\le c\,x^2 \iff
x\ge 4/c$, so **every** $c > 0$ works with $x_0 = 4/c$: a smaller constant just needs a later starting point. The pairs
$(1, 3)$ and $(0.5, 4)$ fail — at $x = 3$, $4x = 12 > 9 = x^2$ — and even the tiny constant $c = 0.001$ fails with
$x_0 = 100$ (it would need $x_0 = 4000$). A failure on the grid *disproves* a candidate pair; a success on a grid is
only evidence, because a grid cannot check every real $x$. The proof is the one-line algebra above.

### 2.2 The picture of a witness

The definition says: from $x_0$ on, the curve of $h$ stays below the curve of $c\cdot g$. The figure shades this region
for three witness pairs.

In [ ]:
# ── 2.2 Three witness pairs for 4x ∈ O(x²): from x0 on, 4x stays under c·x² ──
x = np.linspace(0, 10, 501)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for ax, (c, x0) in zip(axes, [(1, 4), (2, 2), (0.5, 8)]):
    ax.plot(x, 4 * x, color="tab:blue", label="h(x) = 4x")
    ax.plot(x, c * x ** 2, color="tab:red", label=f"c·g(x) = {c}·x²")
    ax.axvspan(x0, 10, color="tab:green", alpha=0.12, label=f"x ≥ x0 = {x0}")
    ax.axvline(x0, color="tab:green", ls="--", lw=1)
    ax.set_ylim(0, 60); ax.set_xlabel("x"); ax.set_title(f"witnesses c = {c}, x0 = {x0}"); ax.legend(loc="upper left")
axes[0].set_ylabel("value")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In every panel the blue line stays below the red curve throughout the shaded region
$x\ge x_0$; to the left of $x_0$ anything may happen (and does: the line is above the curve near 0). Changing $c$
stretches the red curve vertically, which moves the crossing point to $x = 4/c$: $c = 2$ crosses at 2, $c = 0.5$ at 8.
One proof is enough — the set $O(x^2)$ contains $4x$.

### 2.3 Disproving membership: $x^2\notin O(4x)$

To show that $f\notin O(g)$ we negate the definition, pushing ¬ through the quantifiers exactly as in Module 02:

$$\neg\big(\exists x_0\ \exists c\ \forall x\ge x_0: f(x)\le c\,g(x)\big) \iff \forall x_0\ge0\ \forall c>0\ \exists x\ge x_0:\ f(x) > c\,g(x).$$

*In words:* $f\notin O(g)$ means that **whatever** constant and threshold you propose, there is a later input where $f$
beats $c$ times $g$. For $f(x) = x^2$ and $g(x) = 4x$: given any $c$ and $x_0$, take $x = \max(x_0, 4c) + 1$. Then
$x > 4c$, so $x^2 = x\cdot x > 4c\cdot x = c\cdot 4x$. No pair of witnesses can exist.

In [ ]:
# ── 2.3 A "counterexample machine": for ANY proposed (c, x0) it returns an x that breaks x² ≤ c·4x ──
def counterexample_x2_vs_4x(c: float, x0: float) -> float:
    """An x ≥ x0 with x² > c·4x, following the proof: x = max(x0, 4c) + 1."""
    return max(x0, 4 * c) + 1


for c, x0 in [(1, 0), (10, 5), (1000, 2), (1e6, 1e3)]:
    x_bad = counterexample_x2_vs_4x(c, x0)
    print(f"proposed c = {c:<9g} x0 = {x0:<6g} → x = {x_bad:<9g}: x² = {x_bad ** 2:.4g} > c·4x = {c * 4 * x_bad:.4g}? "
          f"{x_bad ** 2 > c * 4 * x_bad}")

🔍 **What just happened?** Every proposal is defeated: even $c = 10^6$ with $x_0 = 1000$ fails at
$x = 4{,}000{,}001$, where $x^2\approx 1.6\times10^{13}$ exceeds $c\cdot 4x\approx 1.6\times10^{13}$ — just barely, as
the proof predicts. This is the general pattern: **to prove $f\in O(g)$, exhibit one witness pair; to prove $f\notin
O(g)$, show how to defeat every pair.** Together, $4x\in O(x^2)$ and $x^2\notin O(4x)$ say that $x^2$ grows *strictly*
faster than $4x$.

✏️ **Your turn.** (a) Find witnesses for $100x + 50\in O(x)$ — for instance with $c = 150$, which $x_0$ works? Check
with `check_witness`. (b) Write a counterexample machine for $x^3\notin O(100x^2)$.

### 📘 Lecture notes — Big-O

**The question.** To understand the behaviour of a function $f(x)$ we compare its growth with another function $g(x)$:
*does $f(x)$ grow no faster than $g(x)$?*

**Definition.** For $h, g: \mathbb R^{\ge0}\to\mathbb R^{\ge0}$,
$O(g(x)) := \{h(x) \mid \exists x_0\ge 0, c>0 : \forall x\ge x_0 : h(x)\le c\cdot g(x)\}$ — a **set** of functions (notice
the explicit use of ≥ vs >). The question becomes: *is $f(x)\in O(g(x))$?*

**Pedantry.** $f(x) = O(g(x))$ is not technically correct (it asks whether a function equals a set of functions);
write $f(x)\in O(g(x))$.

**Worked example.** $f(x) = 4x$, $g(x) = x^2$: $4x\le x^2$ for all $x\ge 4$, so $(c, x_0) = (1, 4)$ are witnesses and
$4x\in O(x^2)$; any $c>0$ works with $x_0 = 4/c$. Conversely $x^2\notin O(4x)$: for any $c, x_0$ the input
$x = \max(x_0, 4c) + 1$ gives $x^2 > c\cdot 4x$.

**Negation.** $f\notin O(g) \iff \forall x_0\ge0\ \forall c>0\ \exists x\ge x_0: f(x) > c\,g(x)$.

**Common confusions.** (i) Big-O is an *upper* bound only: $4x\in O(x^2)$ is true even though $x^2$ is a very generous
bound. (ii) The inequality must hold for **all** $x\ge x_0$, not just at some points or on average. (iii) Witnesses are
not unique.

> 🎓 **Exam tip.** For "is $f\in O(g)$?" either give explicit $(c, x_0)$ and verify $f(x)\le c\,g(x)$ for all
> $x\ge x_0$ with algebra, or negate the definition and show how to defeat any $(c, x_0)$.

---
## 3 · Rules for working with Big-O

> 📍 **Where we are.** Finding witnesses for every function from scratch would be tedious. Four rules let us reason
> about Big-O the way we reason about inequalities — and each one is proved directly from the definition. Throughout,
> all functions are non-negative.

**(P1) A sum behaves like its largest term.**

$$f(x)\in O\big(g(x) + h(x)\big) \iff f(x)\in O\big(\max(g(x), h(x))\big).$$

*In words:* being bounded by the sum of two functions is the same as being bounded by the larger of the two.
*Proof.* For non-negative numbers, $\max(g, h)\le g + h\le 2\max(g, h)$. If $f\le c\,(g+h)$ then $f\le 2c\max(g,h)$;
if $f\le c\max(g,h)$ then $f\le c\,(g+h)$. The same $x_0$ works, with the constant doubled or kept. ∎

**(P2) Transitivity.**

$$f(x)\in O(g(x)) \text{ and } g(x)\in O(h(x)) \implies f(x)\in O(h(x)).$$

*In words:* if $f$ grows no faster than $g$, and $g$ no faster than $h$, then $f$ grows no faster than $h$.
*Proof.* We have $f\le c_1 g$ for $x\ge x_1$ and $g\le c_2 h$ for $x\ge x_2$. For $x\ge\max(x_1, x_2)$ both hold, so
$f\le c_1 g\le c_1 c_2 h$: witnesses $c = c_1c_2$, $x_0 = \max(x_1, x_2)$. ∎

**(P3) Sums.**

$$f(x)\in O(h_1(x)) \text{ and } g(x)\in O(h_2(x)) \implies f(x) + g(x)\in O\big(h_1(x) + h_2(x)\big).$$

*In words:* the cost of doing two things one after the other is bounded by the sum of their bounds.
*Proof.* For $x\ge\max(x_1, x_2)$: $f + g\le c_1h_1 + c_2h_2\le\max(c_1, c_2)\,(h_1 + h_2)$. ∎

**(P4) Products.**

$$f(x)\in O(h_1(x)) \text{ and } g(x)\in O(h_2(x)) \implies f(x)\cdot g(x)\in O\big(h_1(x)\cdot h_2(x)\big).$$

*In words:* doing $g$ amount of work for each of $f$ items costs at most the product of the bounds — this is how
nested loops multiply. *Proof.* For $x\ge\max(x_1, x_2)$, multiplying the two inequalities between non-negative
numbers gives $f g\le (c_1h_1)(c_2h_2) = (c_1c_2)\,h_1h_2$. ∎

**Biological reading.** An RNA-seq pipeline that does quality control in time $O(N)$, sorts the $N$ reads in
$O(N\log N)$ and then compares all pairs of reads in $O(N^2)$ costs, by P3 and P1, $O(N + N\log N + N^2) = O(N^2)$:
the most expensive step decides. The next cell shows how the dominant term takes over.

In [ ]:
# ── 3.1 A pipeline cost model (illustrative constants): which term dominates as N grows? ──
def pipeline_terms(N):
    """Cost of the three pipeline steps (illustrative constants) for N reads."""
    N = np.asarray(N, dtype=float)
    return {"QC: 200·N": 200 * N, "sorting: 3·N·log₂N": 3 * N * np.log2(N), "all pairs: 0.001·N²": 0.001 * N ** 2}


for n_reads in [1e3, 1e5, 1e7, 1e9]:
    parts = pipeline_terms(n_reads)
    total_cost = sum(parts.values())
    shares = ", ".join(f"{name.split(':')[0]} {100 * t / total_cost:5.1f} %" for name, t in parts.items())
    print(f"N = {n_reads:8.0e}: {shares}")

N = np.logspace(1, 9, 200)                          # number of reads, 10 … 10⁹
terms = pipeline_terms(N)
total = sum(terms.values())

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.stackplot(N, *[t / total for t in terms.values()], labels=list(terms), alpha=0.85,
             colors=["tab:green", "tab:orange", "tab:red"])
ax.set_xscale("log"); ax.set_xlabel("number of reads N (log scale)"); ax.set_ylabel("share of the total cost")
ax.set_title("sum ↔ max: eventually the fastest-growing term is (almost) all of the cost")
ax.legend(loc="center left", fontsize=7); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With these illustrative constants, quality control (the largest *constant*, 200 per read)
dominates small runs: at $N = 1{,}000$ it is 86.6 % of the cost and the all-pairs step only 0.4 %. Sorting never
dominates (its share peaks at about 16 % near $N\approx 20{,}000$), because its cost per read, $3\log_2 N$, only reaches
about 90 even at $N = 10^9$ — always below the 200 per read of quality control. The all-pairs step starts tiny
($0.001\cdot N^2$) but already takes 28.6 % at $N = 10^5$, 97.4 % at $10^7$ and, to the printed precision, 100 % at
$10^9$ reads. That is P1 in action: in the long run $200N + 3N\log_2N + 0.001N^2\in O(N^2)$, and the constants only
decide *where* the takeover happens.

✏️ **Your turn.** Change the all-pairs constant from 0.001 to 0.000001 (a much faster comparison). At roughly which
$N$ does the all-pairs step now take over half of the cost? Does the asymptotic conclusion change?

### 3.2 Nested loops multiply (P4)

Most running times are found by counting how often the innermost line of a program runs. A loop over $n$ items that
contains a loop over $m$ items runs its body $n\cdot m$ times — the product rule P4 with $h_1 = n$, $h_2 = m$. When the
inner loop only runs over the items *after* the current one (as in all-pairs comparisons), the count is smaller,
$\binom n2 = n(n-1)/2$, but still at most $n\cdot n$, so it lies in $O(n^2)$.

In [ ]:
# ── 3.2 Count the steps of nested loops and compare with the product rule ──
def count_steps(n: int, m: int, triangular: bool = False) -> int:
    """How many times the body of a double loop runs: all (i, j) pairs, or only j > i (all-pairs comparison)."""
    steps = 0
    for i in range(n):
        for j in (range(i + 1, n) if triangular else range(m)):
            steps += 1
    return steps


for n in [10, 100, 1000]:
    full, tri = count_steps(n, n), count_steps(n, n, triangular=True)
    print(f"n = {n:4d}: full double loop {full:>9,} = n·n | j > i only {tri:>7,} = n(n−1)/2 = {n * (n - 1) // 2:>7,}"
          f" | ratio to n² = {tri / n ** 2:.4f}")
print("reads × k-mers, n = 300, m = 40:", f"{count_steps(300, 40):,}", "= 300 · 40")

🔍 **What just happened?** The full double loop runs exactly $n\cdot n$ times (100, 10,000, 1,000,000), and a loop over
300 reads containing a loop over 40 k-mers runs $300\cdot 40 = 12{,}000$ times — counting confirms the product rule.
The triangular loop of an all-pairs comparison runs $n(n-1)/2$ times (45, 4,950, 499,500); its ratio to $n^2$
approaches 1/2 (0.45, 0.495, 0.4995), so it is in $O(n^2)$ — and, as Section 6 will say, in $\Theta(n^2)$: halving the
work does not change the growth class.

### 📘 Lecture notes — properties of Big-O

For non-negative functions:

* $f(x)\in O(g(x) + h(x)) \iff f(x)\in O(\max(g(x), h(x)))$ — because $\max(g,h)\le g+h\le 2\max(g,h)$.
* $f(x)\in O(g(x))$ and $g(x)\in O(h(x)) \implies f(x)\in O(h(x))$ — witnesses $c_1c_2$, $\max(x_1, x_2)$.
* $f(x)\in O(h_1(x))$ and $g(x)\in O(h_2(x)) \implies f(x) + g(x)\in O(h_1(x) + h_2(x))$ — constant $\max(c_1, c_2)$.
* $f(x)\in O(h_1(x))$ and $g(x)\in O(h_2(x)) \implies f(x)\cdot g(x)\in O(h_1(x)\cdot h_2(x))$ — constant $c_1c_2$.

Also: constant multiples do not matter, $a\cdot f\in O(f)$ for any constant $a > 0$ (witness $c = a$).

**Common confusion.** The rules give upper bounds; they do not say the bound is tight. $x + x^2\in O(x^3)$ is true
(by transitivity) but loose; the tight statement is $\Theta(x^2)$ (Section 6).

> 🎓 **Exam tip.** In a proof, always state the combined witnesses explicitly — typically
> $x_0 = \max(x_1, x_2)$ and $c = c_1c_2$ or $\max(c_1, c_2)$ — and say where you used non-negativity.

---
## 4 · Polynomials: $p(x)\in O(x^k)$

> 📍 **Where we are.** The rules of Section 3 immediately settle the most common case in practice: running times that
> are polynomials, like $3n^2 + 2$ or $\tfrac12 n(n-1)$.

Let

$$p(x) = \sum_{i=0}^{k} a_i x^i = a_0 + a_1x + a_2x^2 + \dots + a_kx^k$$

be a polynomial of degree (order) $k$ — real coefficients $a_i$, with $a_k\ne 0$. *In words:* a polynomial is a sum of
powers of $x$, each multiplied by a fixed number; the degree is the highest power that appears.

**Theorem.** $p(x)\in O(x^k)$.

**Proof idea.** For $x\ge 1$, a smaller power is never larger than a bigger one: $x^i\le x^k$ whenever $i\le k$. Hence,
term by term, $a_ix^i\le|a_i|\,x^i\le|a_i|\,x^k$, and adding the $k+1$ terms,

$$p(x) \;\le\; \big(|a_0| + |a_1| + \dots + |a_k|\big)\,x^k \quad\text{for all } x\ge 1 .$$

*In words:* every term of the polynomial is at most (its coefficient's size) × $x^k$ once $x\ge 1$, so the whole
polynomial is at most (sum of the coefficient sizes) × $x^k$; witnesses $c = \sum_i|a_i|$ and $x_0 = 1$. ∎

⚠️ **Common confusion.** In the sum, the power must follow the index — $a_ix^{\,i}$, not $a_ix^{\,k}$ — and the key
inequality $x^i\le x^k$ holds for $x\ge 1$, not for $0 < x < 1$ (there, $x^2 < x$). Also note that our definition needs
non-negative functions: a polynomial with $a_k > 0$ is positive for all large $x$, which is all the definition looks
at.

**Worked examples from 02-680.**
* **Is $3x^2 + 2\in O(x^2)$?** Yes. For $x\ge 1$, $2\le 2x^2$, so $3x^2 + 2\le 3x^2 + 2x^2 = 5x^2$: witnesses
  $c = 5 = |3| + |2|$, $x_0 = 1$.
* **Is $x^3\in O(x^2)$?** No. If $x^3\le c\,x^2$ for all $x\ge x_0$, dividing by $x^2 > 0$ gives $x\le c$ for all
  $x\ge x_0$ — false for $x = \max(x_0, c) + 1$.

In [ ]:
# ── 4.1 3x² + 2 ∈ O(x²): which constants c work, and from which x0? (exact algebra with SymPy) ──
x_s = sp.symbols("x", positive=True)
p = 3 * x_s ** 2 + 2
for c in [5, 4, sp.Rational(31, 10), 3]:
    # p(x) ≤ c·x²  ⇔  (c − 3)·x² ≥ 2  ⇔  x ≥ sqrt(2 / (c − 3))   (only possible if c > 3)
    if c > 3:
        x0 = sp.sqrt(sp.Rational(2) / (c - 3))
        print(f"c = {str(c):5s}: 3x² + 2 ≤ {c}·x² ⇔ x ≥ √(2/(c−3)) = {sp.nsimplify(x0)} ≈ {float(x0):.3f}  → x0 = {float(x0):.3f}")
    else:
        print(f"c = {str(c):5s}: 3x² + 2 ≤ 3x² would need 2 ≤ 0 — impossible for every x: c must be larger than 3")

print("\nratio (3x² + 2)/x² as x → ∞:", sp.limit(p / x_s ** 2, x_s, sp.oo))
q = 2 * x_s ** 3 - 5 * x_s ** 2 + 7
print("q(x) = 2x³ − 5x² + 7: proof constant Σ|a_i| =", 2 + 5 + 7, "| q(x) ≤ 14x³ for x ≥ 1? (checked on 1…1000)",
      all(q.subs(x_s, v) <= 14 * v ** 3 for v in range(1, 1001)), "| limit of q/x³:", sp.limit(q / x_s ** 3, x_s, sp.oo))
print("x³/x² = x → limit", sp.limit(x_s ** 3 / x_s ** 2, x_s, sp.oo), "→ x³ ∉ O(x²)")

🔍 **What just happened?** For $3x^2 + 2\le c\,x^2$ we need $(c-3)x^2\ge 2$. With the proof's constant $c = 5$ the
threshold is $x_0 = 1$; with $c = 4$ it is $\sqrt 2\approx 1.414$; with $c = 3.1$ it is $\sqrt{20}\approx 4.472$; and
with $c = 3$ no threshold exists, because the "+2" never goes away. So every constant above the leading coefficient 3
works, with a later start the closer $c$ gets to 3. SymPy confirms that the ratio $(3x^2+2)/x^2$ tends to 3 — the
leading coefficient. For $q(x) = 2x^3 - 5x^2 + 7$ the proof gives $c = 2 + 5 + 7 = 14$ (generous but valid), and the
ratio $q/x^3$ tends to 2. Finally $x^3/x^2 = x\to\infty$: no constant can bound it, so $x^3\notin O(x^2)$.

🧮 **Math refresher — limits, informally.** $\lim_{x\to\infty} r(x) = L$ means that $r(x)$ gets as close to $L$ as you
like once $x$ is large enough; $\lim_{x\to\infty} r(x) = \infty$ means $r(x)$ eventually exceeds any number you name.
(Module 14 treats limits carefully.) Limits of **ratios** give a quick test:

$$\lim_{x\to\infty}\frac{f(x)}{g(x)} = L < \infty \;\implies\; f\in O(g), \qquad \lim_{x\to\infty}\frac{f(x)}{g(x)} = \infty \;\implies\; f\notin O(g).$$

*In words:* if the ratio settles down to a finite number $L$, then eventually $f\le (L+1)\,g$ (witness $c = L + 1$);
if the ratio grows without bound, no constant can keep $f$ below $c\,g$. Section 6 shows that the test can be
silent — the limit may not exist.

In [ ]:
# ── 4.2 Ratios p(x)/x^k settle to the leading coefficient; x³/x² does not settle ──
x = np.logspace(0, 4, 300)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].semilogx(x, (3 * x ** 2 + 2) / x ** 2, label="(3x² + 2) / x² → 3")
axes[0].semilogx(x, (2 * x ** 3 - 5 * x ** 2 + 7) / x ** 3, label="(2x³ − 5x² + 7) / x³ → 2")
axes[0].semilogx(x, (0.5 * x * (x - 1)) / x ** 2, label="x(x−1)/2 / x² → 1/2  (number of pairs)")
axes[0].set_ylim(-1, 6); axes[0].set_xlabel("x (log scale)"); axes[0].set_ylabel("ratio"); axes[0].legend(fontsize=7)
axes[0].set_title("polynomial / x^degree → leading coefficient")
axes[1].loglog(x, x ** 3 / x ** 2, color="tab:red", label="x³ / x² = x → ∞")
axes[1].set_xlabel("x (log scale)"); axes[1].set_ylabel("ratio (log scale)"); axes[1].legend()
axes[1].set_title("x³ / x² grows without bound: x³ ∉ O(x²)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each polynomial divided by its highest power flattens out at its leading coefficient — 3, 2
and 1/2 — after some wiggles at small $x$ (the cubic ratio dips because of $-5x^2$, and at $x = 1$ it equals
$2 - 5 + 7 = 4$). The third curve is the number of pairs among $x$ objects, $\binom x2 = x(x-1)/2$: all-pairs comparison
is $O(x^2)$ with constant 1/2. The ratio $x^3/x^2 = x$ on the right is a straight line on log–log axes heading to
infinity. So for polynomials Big-O just reads off the degree: **$p\in O(x^k)$ iff $k\ge$ the degree of $p$.**

✏️ **Your turn.** (a) Find witnesses for $\tfrac12 x(x-1)\in O(x^2)$ by hand (try $c = \tfrac12$, $x_0 = 0$). (b) Is
$x^2\in O(x^3 - 100x^2)$? Careful: the right-hand side is negative for $x < 100$ — what does that mean for $x_0$?

---
## 5 · Logarithms ≪ polynomials ≪ exponentials

> 📍 **Where we are.** Running times of real algorithms involve logarithms (binary search, sorting), polynomials (dynamic
> programming) and exponentials (exhaustive search). Their ordering is the single most useful fact of asymptotics:
> **log ≪ polynomial ≪ exponential**.

**Logarithms are below every positive power.** For every $\epsilon > 0$ (however small),

$$\log x\in O(x^{\epsilon}) .$$

*In words:* a logarithm grows more slowly than any root of $x$ — even $x^{0.01}$ eventually overtakes $\log x$.
(A logarithm is undefined at 0 and negative below 1, so for logarithms we only look at inputs $x\ge 1$, which is all
the "eventually" in the definition needs.)

**The base of the logarithm does not matter.** Because $\log_b x = \dfrac{\ln x}{\ln b}$, two logarithms differ by the
constant factor $\ln b$, so $O(\log_2 x) = O(\ln x) = O(\log_{10} x)$ — which is why one simply writes $O(\log x)$.

**Polynomials are below every exponential.** For the polynomial $p$ of Section 4 and every base $b > 1$,

$$p(x)\in O(b^x).$$

*In words:* any polynomial, of any degree, is eventually overtaken by any exponential with base larger than 1 — even
$1.0001^x$ eventually beats $x^{100}$.

**The base of the exponential *does* matter.** For bases $b, c > 0$,

$$b^x\in O(c^x) \iff b\le c .$$

*In words:* one exponential is bounded by another exactly when its base is not larger. *Proof.* If $b\le c$ then
$b^x\le c^x$ for all $x\ge 0$ (witnesses $1, 0$). If $b > c$, the ratio $b^x/c^x = (b/c)^x$ with $b/c > 1$ grows without
bound, so no constant works (the limit test of Section 4). ∎ For example $4^x = (2^2)^x = (2^x)^2\notin O(2^x)$ — unlike
logarithms, where $\log(x^2) = 2\log x\in O(\log x)$.

In [ ]:
# ── 5.1 "Eventually" can be very late: where do log x and x^0.1, and x^10 and 1.1^x, cross? ──
# ln x = x^0.1 ⇔ (with x = e^t) t = e^(0.1 t): solve for t with a root finder, then x = e^t
t_low = brentq(lambda t: t - math.exp(0.1 * t), 0.5, 5)
t_high = brentq(lambda t: t - math.exp(0.1 * t), 10, 100)
print(f"ln x = x^0.1 at x ≈ {math.exp(t_low):.3f} and at x ≈ {math.exp(t_high):.3e}")
print(f"  e.g. x = 1e6: ln x = {math.log(1e6):.2f} > x^0.1 = {1e6 ** 0.1:.2f};  x = 1e16: ln x = {math.log(1e16):.2f} < x^0.1 = {1e16 ** 0.1:.2f}")

x_cross = brentq(lambda v: 10 * math.log(v) - v * math.log(1.1), 10, 5000)        # x^10 = 1.1^x
print(f"\nx^10 = 1.1^x at x ≈ {x_cross:.1f}; at x = 1000: log10(x^10) = {10 * math.log10(1000):.0f}, "
      f"log10(1.1^x) = {1000 * math.log10(1.1):.1f}")
for k in [2, 3, 10]:
    xc = brentq(lambda v: k * math.log(v) - v * math.log(2), k + 0.5, 1e4)
    print(f"x^{k:<2d} = 2^x last crosses at x ≈ {xc:.2f}")

print("\nbase of the log: log₂ x / ln x =", round(math.log2(1000) / math.log(1000), 6), "= 1/ln 2 =", round(1 / math.log(2), 6),
      "| log10 x / ln x =", round(math.log10(1000) / math.log(1000), 6))
for b, c in [(2, 3), (3, 2), (2, 2)]:
    ratios = [(b / c) ** v for v in (10, 100, 1000)]
    print(f"b = {b}, c = {c}: (b/c)^x at x = 10, 100, 1000 → {', '.join(f'{r:.3g}' for r in ratios)}  ⇒ b^x ∈ O(c^x)? {b <= c}")

🔍 **What just happened?** $\ln x$ and $x^{0.1}$ cross twice: $\ln x$ overtakes $x^{0.1}$ at $x\approx 3.06$ and only falls
back below it at $x\approx 3.4\times10^{15}$ — at a million, $\ln x = 13.8$ is still far above $x^{0.1} = 3.98$. The
theorem $\log x\in O(x^{0.1})$ is nevertheless true (witnesses $c = 1$, $x_0 = 3.5\times10^{15}$): asymptotics promises
"eventually", and eventually can be astronomically late. Likewise $x^{10}$ beats $1.1^x$ until $x\approx 685$; by
$x = 1000$ the exponential is ahead by more than ten orders of magnitude ($10^{41.4}$ vs $10^{30}$). Against $2^x$,
$x^2$ last crosses at 4, $x^3$ at about 9.94 and $x^{10}$ at about 58.8. The logarithm ratios are constants
($1/\ln 2\approx 1.4427$ and $1/\ln 10\approx 0.4343$), while $(b/c)^x$ explodes for $b = 3 > c = 2$, vanishes for
$b < c$ and stays 1 for $b = c$ — the theorem $b^x\in O(c^x)\iff b\le c$ in numbers.

### 5.2 The growth hierarchy in one picture

On a logarithmic vertical axis, every function of the zoo — plus the factorial $x!$ — can be drawn together.

In [ ]:
# ── 5.2 The hierarchy log ≪ √ ≪ linear ≪ x log x ≪ x² ≪ x³ ≪ 2^x ≪ x! on a log scale ──
x = np.arange(1, 41, dtype=float)
fig, ax = plt.subplots(figsize=(6.6, 4.0))
for name, f in GROWTH.items():
    ax.semilogy(x, np.maximum(f(x), 1e-1), label=name)
fact = np.array([math.factorial(int(v)) for v in x], dtype=float)
ax.semilogy(x, fact, "k--", label="x!")
ax.axhline(1e9, color="0.5", ls=":", lw=1); ax.text(1, 2e9, "10⁹ operations ≈ 1 s of computing", fontsize=7, color="0.4")
ax.set_ylim(0.1, 1e30); ax.set_xlabel("input size x"); ax.set_ylabel("f(x) (log scale)")
ax.set_title("log ≪ polynomial ≪ exponential ≪ factorial"); ax.legend(ncol=2, fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

ops_per_second = 1e9
budget = {"log₂ n": lambda n: math.log2(n), "n": lambda n: n, "n log₂ n": lambda n: n * math.log2(n),
          "n²": lambda n: n ** 2, "n³": lambda n: n ** 3, "2^n": lambda n: 2.0 ** n,
          "n!": lambda n: float(math.factorial(n)) if n < 171 else math.inf}


def largest_n(cost, limit: float = ops_per_second) -> int:
    """Largest whole n ≥ 2 with cost(n) ≤ limit (doubling, then binary search)."""
    lo, hi = 2, 4
    while cost(hi) <= limit and hi < 10 ** 300:
        lo, hi = hi, hi * 2
    while hi - lo > 1:
        mid = (lo + hi) // 2
        lo, hi = (mid, hi) if cost(mid) <= limit else (lo, mid)
    return lo


for name, cost in budget.items():
    n_max = largest_n(cost)
    print(f"{name:9s}: largest n doable with 10⁹ operations ≈ {n_max:.3g}" if n_max > 1e15
          else f"{name:9s}: largest n doable with 10⁹ operations = {n_max:,}")

🔍 **What just happened?** In the picture the curves fan out in the predicted order, and the dashed factorial overtakes
even $2^x$ (from $x = 4$ on). The table asks the practical question: with $10^9$ elementary operations (roughly a
second of computing), how large can the input be? Linear: a billion. $n\log_2 n$: about 40 million ($39{,}620{,}077$).
$n^2$: 31,622. $n^3$: 1,000. $2^n$: 29. $n!$: 12. For $\log_2 n$ the search stops at the doubling cap ($\approx10^{300}$)
— a logarithmic algorithm could handle inputs larger than the number of atoms in the universe. Buying a computer 1,000
times faster turns 31,622 into 1,000,000 for the quadratic method, but turns 29 into only 39 for the exponential one
($2^{10} = 1024$): **for exponential algorithms, faster hardware hardly helps.**

✏️ **Your turn.** Repeat the table for a budget of $10^{12}$ operations (about 20 minutes). Which rows change by a factor
of about 1,000, which by about 30, which by exactly 10, and which only by a handful of *additions* to $n$? Explain each
change from the formula.

### 📘 Lecture notes — logarithms, polynomials and exponentials

**Polynomials.** Let $p(x) = \sum_{i=0}^{k} a_ix^i$ be a polynomial of order $k$. Then $p(x)\in O(x^k)$. *Basic idea of
proof:* for $i<k$ and $x > 1$, $a_ix^i\le|a_i|\,x^k$, so $p(x)\le(\sum_i|a_i|)\,x^k$ for $x\ge 1$.

**The hierarchy.** Log ≪ Polynomial ≪ Exponential:
* $\log x\in O(x^{\epsilon})$ for every $\epsilon > 0$ (the base of the log does not matter: $\log_bx = \ln x/\ln b$);
* $p(x)\in O(b^x)$ for any $b > 1$ and any polynomial $p$ of degree $k\ge 0$;
* $b^x\in O(c^x)\iff b\le c$ (for $b, c > 0$): the base of an exponential matters.

**Worked examples.** $3x^2 + 2\in O(x^2)$ (witnesses $c = 5$, $x_0 = 1$); $x^3\notin O(x^2)$ ($x^3\le cx^2\iff x\le c$
fails for large $x$); $\ln x\le x^{0.1}$ only from $x\approx 3.4\times10^{15}$ on; $x^{10}\le 1.1^x$ from $x\approx 685$ on.

**Common confusions.** (i) "Eventually" may mean astronomically large $x$ — asymptotics does not tell you *where*
the crossing is. (ii) $4^x\notin O(2^x)$ even though both are "exponential". (iii) The proof inequality is
$a_ix^{\,i}\le|a_i|x^{\,k}$ for $x\ge 1$ (power $i$ on the left, and $x$ — not $n$ — as the variable).

> 🎓 **Exam tip.** To rank a list of functions, simplify each to its dominant term, then use: logs (any base) <
> powers (by exponent) < exponentials (by base) < factorials. Justify each comparison with a limit of the ratio.

---
## 6 · The whole family: Ω, Θ, o and ω

> 📍 **Where we are.** Big-O is an upper bound ("no faster than"). To say "at least as fast", "exactly as fast" or
> "strictly faster", 02-680 defines four more sets with the same quantifier pattern — only the inequality and one
> quantifier change.

**Big-Omega (lower bound).**

$$\Omega\big(g(x)\big) := \Big\{h(x) \;\Big|\; \exists x_0\ge 0,\ d > 0:\ \forall x\ge x_0:\ h(x) \ge d\cdot g(x)\Big\}.$$

*In words:* $h\in\Omega(g)$ if, from some point on, $h$ is **at least** a fixed positive fraction $d$ of $g$ — $h$
grows at least as fast as $g$.

**Duality.**

$$f(x)\in O(g(x)) \iff g(x)\in\Omega(f(x)).$$

*In words:* "$f$ grows no faster than $g$" and "$g$ grows at least as fast as $f$" are the same statement.
*Proof.* $f(x)\le c\,g(x) \iff g(x)\ge\tfrac1c f(x)$; keep the same $x_0$ and take $d = 1/c$ (and conversely
$c = 1/d$). ∎

**Big-Theta (same growth rate).**

$$\Theta\big(g(x)\big) := \Big\{h(x) \;\Big|\; h(x)\in O(g(x)) \wedge h(x)\in\Omega(g(x))\Big\} = O(g)\cap\Omega(g).$$

*In words:* $h\in\Theta(g)$ if $h$ is eventually sandwiched between two constant multiples of $g$:
$d\,g(x)\le h(x)\le c\,g(x)$. Example: $3x^2 + 2\in\Theta(x^2)$, while $4x\in O(x^2)$ but $4x\notin\Theta(x^2)$.

**Little-o (strictly slower).**

$$o\big(g(x)\big) := \Big\{h(x) \;\Big|\; \forall c > 0:\ \exists x_0\ge 0:\ \forall x\ge x_0:\ h(x) < c\cdot g(x)\Big\}.$$

*In words:* $h\in o(g)$ if, **for every** positive constant — however tiny — $h$ eventually drops below that fraction of
$g$. The change from "∃ c" to "∀ c" is the whole difference from Big-O: $h$ becomes a negligible fraction of $g$.

**Little-omega (strictly faster).**

$$\omega\big(g(x)\big) := \Big\{h(x) \;\Big|\; \forall c > 0:\ \exists x_0\ge 0:\ \forall x\ge x_0:\ h(x) > c\cdot g(x)\Big\}.$$

*In words:* $h\in\omega(g)$ if $h$ eventually exceeds **every** constant multiple of $g$.

| Set | Analogy | Example with $g = x^2$ |
|---|---|---|
| $O(g)$ | $h\le g$ | $4x$, $3x^2+2$ |
| $\Omega(g)$ | $h\ge g$ | $3x^2+2$, $x^3$ |
| $\Theta(g)$ | $h = g$ | $3x^2+2$, $\tfrac12x(x-1)$ |
| $o(g)$ | $h < g$ | $4x$, $x\log x$ |
| $\omega(g)$ | $h > g$ | $x^3$, $2^x$ |

For functions whose ratio has a limit, the limit decides: $\lim f/g = 0 \Rightarrow f\in o(g)$;
$0 < \lim f/g < \infty \Rightarrow f\in\Theta(g)$; $\lim f/g = \infty\Rightarrow f\in\omega(g)$.

In [ ]:
# ── 6.1 Classify pairs of functions with the limit of their ratio (SymPy) ──
def classify_growth(f, g, var=x_s) -> str:
    """Compare f and g (SymPy expressions in var) through L = lim f/g as var → ∞; assumes the limit exists."""
    L = sp.limit(f / g, var, sp.oo)
    if L == 0:
        return f"lim f/g = 0 → f ∈ o(g) (hence O(g)), not Ω(g)"
    if L == sp.oo:
        return f"lim f/g = ∞ → f ∈ ω(g) (hence Ω(g)), not O(g)"
    return f"lim f/g = {sp.nsimplify(L)} → f ∈ Θ(g)"


pairs = [(3 * x_s ** 2 + 2, x_s ** 2), (4 * x_s, x_s ** 2), (x_s ** 3, x_s ** 2), (x_s * sp.log(x_s), x_s ** sp.Rational(3, 2)),
         (sp.log(x_s, 2), sp.log(x_s)), (x_s ** 10, sp.Rational(11, 10) ** x_s), (4 ** x_s, 2 ** x_s),
         (x_s * (x_s - 1) / 2, x_s ** 2), (sp.factorial(x_s), 2 ** x_s)]
for f, g in pairs:
    print(f"f = {str(f):18s} g = {str(g):12s}: {classify_growth(f, g)}")

🔍 **What just happened?** SymPy computed each limit exactly. $3x^2+2$ vs $x^2$ → 3: same growth (Θ). $4x$ vs $x^2$ → 0:
strictly slower (o). $x^3$ vs $x^2$ → ∞: strictly faster (ω). $x\log x$ vs $x^{3/2}$ → 0: a logarithm is below the
power $x^{1/2}$ (the hierarchy again). $\log_2x$ vs $\ln x$ → $1/\log 2$: Θ — the base of a log does not matter.
$x^{10}$ vs $1.1^x$ → 0: polynomial below exponential. $4^x$ vs $2^x$ → ∞: the base of an exponential does matter.
The number of pairs $x(x-1)/2$ is $\Theta(x^2)$ with limit 1/2, and the factorial outgrows $2^x$.

The picture of Θ is a **sandwich**: from $x_0$ on, $h$ lies between $d\cdot g$ and $c\cdot g$.

In [ ]:
# ── 6.1b The Θ sandwich: 3·x² ≤ 3x² + 2 ≤ 5·x² for x ≥ 1, while 4x is not sandwiched ──
x = np.linspace(0.01, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].fill_between(x, 3 * x ** 2, 5 * x ** 2, color="tab:green", alpha=0.15, label="between 3·x² and 5·x²")
axes[0].plot(x, 3 * x ** 2 + 2, color="tab:blue", label="h(x) = 3x² + 2")
axes[0].axvline(1, color="0.4", ls=":", label="x0 = 1")
axes[0].set_ylim(0, 120); axes[0].set_xlabel("x"); axes[0].set_ylabel("value"); axes[0].legend(fontsize=7)
axes[0].set_title("3x² + 2 ∈ Θ(x²): witnesses d = 3, c = 5, x0 = 1")
axes[1].plot(x, 4 * x / x ** 2, color="tab:orange", label="4x / x² = 4/x")
axes[1].plot(x, (3 * x ** 2 + 2) / x ** 2, color="tab:blue", label="(3x² + 2) / x²")
axes[1].axhspan(3, 5, color="tab:green", alpha=0.15)
axes[1].set_ylim(0, 8); axes[1].set_xlabel("x"); axes[1].set_ylabel("ratio to x²"); axes[1].legend(fontsize=7)
axes[1].set_title("Θ: the ratio stays in a band away from 0; o: it → 0")
plt.tight_layout(); plt.show()
ratios = (3 * np.logspace(0, 6, 1000) ** 2 + 2) / np.logspace(0, 6, 1000) ** 2
print(f"(3x² + 2)/x² on 1 ≤ x ≤ 10⁶: min {ratios.min():.4f}, max {ratios.max():.4f}")

🔍 **What just happened?** For $x\ge 1$ the blue curve $3x^2 + 2$ stays inside the green band between $3x^2$ and $5x^2$
(left), so $d = 3$, $c = 5$, $x_0 = 1$ are witnesses for $\Theta(x^2)$; numerically its ratio to $x^2$ stays between 3
(approached for large $x$) and 5 (reached at $x = 1$). On the right, the ratio of $4x$ to $x^2$ sinks to 0 and
leaves every band $[d, c]$ with $d > 0$: $4x\in O(x^2)$ but $4x\notin\Omega(x^2)$, hence $4x\notin\Theta(x^2)$ — it is
in $o(x^2)$.

### 6.2 When the limit test is silent

The limit test assumes that $\lim f/g$ exists. It need not. A function can wobble so that its ratio to $g$ has no limit,
and then strange things happen. Here is a warning example: let $g(x) = x^2$ and let $h$ be defined on whole numbers by
$h(x) = x^2$ for even $x$ and $h(x) = x$ for odd $x$.

* $h\in O(g)$: $h(x)\le x^2$ always (witnesses $c = 1$, $x_0 = 0$).
* $h\notin\Omega(g)$: for odd $x$, $h(x) = x\ge d\,x^2$ fails as soon as $x > 1/d$.
* $h\notin o(g)$: with $c = \tfrac12$, every even $x$ has $h(x) = x^2\not<\tfrac12x^2$.

⚠️ **Common confusion.** It is tempting to *define* $o(g)$ as "$O(g)$ but not $\Omega(g)$", and $\omega(g)$ as
"$\Omega(g)$ but not $O(g)$". For well-behaved functions — whenever $\lim f/g$ exists — these descriptions agree with the
"∀ c" definitions. In general they do not: this $h$ lies in $O(g)$ and not in $\Omega(g)$, yet it is **not** in $o(g)$.
The ∀-$c$ definitions are the official ones; the shortcut $o(g) = O(g)\setminus\Omega(g)$ is only safe when the limit of
the ratio exists. (What *is* always true, for positive $g$: $o(g)\subseteq O(g)\setminus\Omega(g)$.)

In [ ]:
# ── 6.2 A wobbling function: in O(x²), not in Ω(x²) — and still not in o(x²) ──
def h_wobble(x: int) -> int:
    """x² for even x, x for odd x (defined on whole numbers)."""
    return x ** 2 if x % 2 == 0 else x


xs_int = np.arange(1, 41)
ratio = np.array([h_wobble(int(v)) / v ** 2 for v in xs_int])
print("h(x)/x² for x = 1…10:", np.round(ratio[:10], 3))
print("in O(x²) with c = 1, x0 = 0?        ", all(h_wobble(int(v)) <= v ** 2 for v in range(0, 10_001)))
print("some d = 0.01 works for Ω(x²) beyond x0 = 1000?", all(h_wobble(int(v)) >= 0.01 * v ** 2 for v in range(1000, 10_001)))
print("c = 1/2 works for o(x²) beyond x0 = 1000?      ", all(h_wobble(int(v)) < 0.5 * v ** 2 for v in range(1000, 10_001)))

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(xs_int, ratio, "o-", ms=3, color="tab:purple")
ax.set_xlabel("x (whole numbers)"); ax.set_ylabel("h(x) / x²")
ax.set_title("the ratio keeps jumping between 1 (even x) and 1/x (odd x): no limit")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The ratio $h(x)/x^2$ alternates: 1, 1, 0.333, 1, 0.2, 1, … — it equals 1 at every even $x$
and $1/x$ at every odd $x$, so it has no limit. The three checks (on whole numbers up to 10,000, which illustrates but
does not prove) agree with the proofs above: $h$ is bounded by $x^2$ (True), no small fraction $d$ bounds it from below
because of the odd $x$ (False), and it never becomes negligible because of the even $x$ (False). Such wobbling running
times are not exotic: an algorithm can be fast on "lucky" inputs and slow on others, which is why algorithm analysis
often speaks of the *worst case*.

✏️ **Your turn.** Use `classify_growth` on $f = x^2 + 100x\log x$ and $g = x^2$, then on $f = \sqrt{x}$ and
$g = \log x^{10}$. Predict the answers first.

### 📘 Lecture notes — Ω, Θ, o and ω

$$\Omega(g) := \{h \mid \exists x_0\ge0, d>0: \forall x\ge x_0: h(x)\ge d\cdot g(x)\}, \qquad f\in O(g)\iff g\in\Omega(f),$$

$$\Theta(g) := \{h \mid h\in O(g)\wedge h\in\Omega(g)\},$$

$$o(g) := \{h \mid \forall c>0: \exists x_0\ge0: \forall x\ge x_0: h(x) < c\cdot g(x)\},$$

$$\omega(g) := \{h \mid \forall c>0: \exists x_0\ge0: \forall x\ge x_0: h(x) > c\cdot g(x)\}.$$

*In words:* Ω = eventually at least a constant fraction of $g$; Θ = sandwiched between two multiples of $g$; o = below
**every** positive multiple of $g$ eventually (negligible); ω = above every multiple of $g$ eventually.

**Limit test** (when $L = \lim f/g$ exists): $L = 0\Rightarrow o$, $0 < L < \infty\Rightarrow\Theta$,
$L = \infty\Rightarrow\omega$.

**Common confusion.** The descriptions "$o(g) = O(g)\wedge\notin\Omega(g)$" and "$\omega(g) = \notin O(g)\wedge\Omega(g)$"
are only correct when the limit of the ratio exists; the counterexample $h(x) = x^2$ (even $x$), $x$ (odd $x$) is in
$O(x^2)\setminus\Omega(x^2)$ but not in $o(x^2)$. Use the ∀-$c$ definitions.

> 🎓 **Exam tip.** Notice which quantifier changes: O and Ω say "∃ c", o and ω say "∀ c". To show $f\notin o(g)$ you
> only need **one** bad constant $c$ (and infinitely many bad $x$).

---
## 7 · Worked exercises

> 📍 **Where we are.** We have the complete toolbox. 02-680 tests it on three useful examples: a function of **two**
> variables, a **wobbling** function, and a comparison of **continuous and discrete** exponential growth.

### 7.1 Is $3x + 4y + 5xy$ in $O(x)$? In $O(y)$? In $O(xy)$?

With two input sizes (say $x$ = number of reads and $y$ = read length), Big-O needs both to be large:

$$h(x, y)\in O(g(x, y)) \iff \exists c > 0,\ x_0\ge 0:\ \forall x\ge x_0,\ \forall y\ge x_0:\ h(x, y)\le c\cdot g(x, y).$$

*In words:* one constant must work for **all** combinations of large $x$ and large $y$ — not just when one of them grows.

* **$O(xy)$: yes.** For $x, y\ge 1$: $3x\le 3xy$ and $4y\le 4xy$, so $3x + 4y + 5xy\le 12\,xy$ (witnesses $c = 12$,
  $x_0 = 1$).
* **$O(x)$: no.** Suppose $3x + 4y + 5xy\le c\,x$ for all $x, y\ge x_0$. Fix any $x\ge\max(x_0, 1)$ and let $y$ grow:
  the left side is at least $5xy$, and $5xy > c\,x$ as soon as $y > c/5$. Contradiction.
* **$O(y)$: no**, by the same argument with the roles of $x$ and $y$ swapped.

But: if $y$ is a **constant** (not an input that grows), then $3x + 4y + 5xy = (3 + 5y)\,x + 4y\in O(x)$. Aligning reads
of a fixed length of 150 bases against a genome of length $n$ costs $O(n\cdot 150) = O(n)$ — true, yet the hidden
constant 150 matters a lot in practice.

In [ ]:
# ── 7.1 The ratios h/(xy), h/x and h/y for h = 3x + 4y + 5xy along different directions ──
def h_two(x, y):
    """The two-variable function 3x + 4y + 5xy."""
    return 3 * x + 4 * y + 5 * x * y


grid = np.logspace(0, 4, 200)
X, Y = np.meshgrid(grid, grid)
print("max of h/(xy) over the grid 1 ≤ x, y ≤ 10⁴:", round(float((h_two(X, Y) / (X * Y)).max()), 3), "(≤ 12, as proved)")
for x_fixed in [1, 10, 100]:
    y_big = 1e6
    print(f"x = {x_fixed:>3}, y = 10⁶: h/x = {h_two(x_fixed, y_big) / x_fixed:,.0f}  (grows with y → not O(x))")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
im = axes[0].pcolormesh(X, Y, h_two(X, Y) / (X * Y), shading="auto", cmap="viridis", vmin=5, vmax=12)
axes[0].set_xscale("log"); axes[0].set_yscale("log"); axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title("h / (xy): bounded (between 5 and 12)"); plt.colorbar(im, ax=axes[0])
im2 = axes[1].pcolormesh(X, Y, np.log10(h_two(X, Y) / X), shading="auto", cmap="magma")
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
axes[1].set_title("log₁₀ (h / x): unbounded as y grows"); plt.colorbar(im2, ax=axes[1])
plt.tight_layout(); plt.show()

🔍 **What just happened?** Over the whole grid $1\le x, y\le 10^4$ the ratio $h/(xy)$ never exceeds 12 (its maximum,
exactly 12, is at $x = y = 1$) and it settles near 5 when both inputs are large: $h\in O(xy)$, in fact $\Theta(xy)$. The
ratio $h/x$, by contrast, is between 5.0 and 9.0 million when $y = 10^6$ (9,000,003 for $x = 1$, 5,400,003 for $x = 10$,
5,040,003 for $x = 100$) and it keeps growing with $y$ (right panel, colour = $\log_{10}$ of the ratio, brightening
upwards). So $h\notin O(x)$, and symmetrically $h\notin O(y)$.

### 7.2 The even/odd function

What is the relationship between $f(x) = x^2$ and

$$g(x) = \begin{cases} x^3 & x \text{ even} \\ x & x \text{ odd}\end{cases}\qquad (x\in\mathbb Z^{\ge 0})?$$

*In words:* $g$ jumps between the cubic $x^3$ (even inputs) and the line $x$ (odd inputs); "$x\,\%\,2 = 0$" in the
notation of programming means "$x$ is even". Is $f\in O(g)$? $\Omega(g)$? $o(g)$? $\omega(g)$? Think before reading on.

* $f\notin O(g)$: for odd $x$, $x^2\le c\,x$ fails as soon as $x > c$.
* $f\notin\Omega(g)$: for even $x$, $x^2\ge d\,x^3$ fails as soon as $x > 1/d$.
* Hence $f\notin o(g)$ (which would require $f\in O(g)$) and $f\notin\omega(g)$ (which would require $f\in\Omega(g)$),
  and also $f\notin\Theta(g)$.

**None of the relations holds**: $f$ and $g$ are *incomparable*. Unlike numbers, functions do not always satisfy
"less, equal or greater".

In [ ]:
# ── 7.2 f(x) = x² against g(x) = x³ (even x) or x (odd x) ──
def g_even_odd(x: int) -> int:
    """x³ if x is even (x % 2 == 0), x if x is odd."""
    return x ** 3 if x % 2 == 0 else x


xs_int = np.arange(1, 31)
ratio_fg = np.array([v ** 2 / g_even_odd(int(v)) for v in xs_int], dtype=float)
print("f/g at x = 1…8:", np.round(ratio_fg[:8], 3))
print("f/g at x = 29 (odd) and 30 (even):", ratio_fg[28], "and", round(ratio_fg[29], 4))

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogy(xs_int[xs_int % 2 == 1], ratio_fg[xs_int % 2 == 1], "o", color="tab:red", label="odd x: f/g = x → ∞")
ax.semilogy(xs_int[xs_int % 2 == 0], ratio_fg[xs_int % 2 == 0], "s", color="tab:blue", label="even x: f/g = 1/x → 0")
ax.axhline(1, color="0.5", ls=":")
ax.set_xlabel("x"); ax.set_ylabel("f(x) / g(x) (log scale)"); ax.legend()
ax.set_title("f = x² and the even/odd g are incomparable: the ratio escapes in both directions")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The ratio $f/g$ equals $x$ for odd $x$ (1, 3, 5, …, 29) and $1/x$ for even $x$ (0.5, 0.25, …,
0.0333 at $x = 30$). On the log scale the red points climb towards ∞ and the blue points sink towards 0. No constant
bounds the ratio from above (so $f\notin O(g)$) and no positive constant bounds it from below (so $f\notin\Omega(g)$):
all four relations O, Ω, o, ω fail.

### 7.3 Continuous vs discrete growth: if α ≈ β, which grows faster?

Two classic models of a growing population (Module 14 develops both):

| Continuous growth | Discrete generations |
|---|---|
| $\dfrac{dy}{dx} = \alpha\,y$ | $y(x+1) = \beta\,y(x)$ |
| $y(x) = e^{\alpha x}\,y(0)$ | $y(x) = \beta^{x}\,y(0)$ |

*In words:* on the left, the population grows at every instant at a rate proportional to its size (bacteria dividing
asynchronously); on the right, it is multiplied by β once per generation (annual plants or insects with
non-overlapping generations). ⚠️ The discrete solution is $\beta^{x}y(0)$, not $\beta^{x+1}y(0)$: at $x = 0$ the formula
must return $y(0)$, and $\beta^x$ does ($\beta^0 = 1$).

Both are exponentials, so Section 5 applies: write $e^{\alpha x} = (e^\alpha)^x$ and compare the **bases** $e^\alpha$
and $\beta$. If the numbers are equal, $\alpha = \beta$, then $e^\alpha > \alpha$ (indeed $e^\alpha\ge 1 + \alpha$), so
$\beta^x\in o(e^{\alpha x})$: the continuous model grows faster. A fairer comparison matches the growth per generation,
$\beta = 1 + r$ with continuous rate $\alpha = r$: since $e^r > 1 + r$ for $r\neq 0$, continuous growth still wins —
compounding during the generation adds a little extra growth, and over many generations a little becomes a lot.

In [ ]:
# ── 7.3 Continuous e^(αx) vs discrete β^x: compare the bases e^α and β ──
r = 0.5
for label, base_cont, base_disc in [("α = β = 0.5 (same number)", math.exp(r), r),
                                     ("α = r = 0.5, β = 1 + r = 1.5", math.exp(r), 1 + r)]:
    print(f"{label:30s}: e^α = {base_cont:.4f} vs β = {base_disc:.4f} → ratio after 20 steps "
          f"= {(base_cont / base_disc) ** 20:.4g}")

steps = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogy(steps, np.exp(r * steps), "o-", label="continuous: y = e^(0.5x)")
ax.semilogy(steps, (1 + r) ** steps, "s-", label="discrete: y = 1.5^x")
ax.set_xlabel("time (generations)"); ax.set_ylabel("population y(x)/y(0) (log scale)"); ax.legend()
ax.set_title("both exponential; the larger base wins: e^0.5 ≈ 1.649 > 1.5")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With $\alpha = \beta = 0.5$ the comparison is lopsided: the discrete "population" is multiplied
by 0.5 per step and *shrinks*, while $e^{0.5}\approx 1.6487$ grows — after 20 steps the ratio is about $2.3\times10^{10}$.
In the fair comparison ($\beta = 1.5$) the continuous model still wins by a factor of about 6.6 after 20 generations,
and on the log scale the two straight lines visibly diverge: their slopes are the logarithms of the bases,
$\ln 1.6487 = 0.5$ vs $\ln 1.5\approx 0.405$. By the theorem $b^x\in O(c^x)\iff b\le c$, $1.5^x\in O(e^{0.5x})$ but not
the other way round.

✏️ **Your turn.** For which $\beta$ does the discrete model grow exactly as fast as $e^{0.5x}$? (Solve $\beta = e^{0.5}$.)
Find the $\beta$ that matches a bacterial culture doubling once per generation in continuous time ($\alpha = \ln 2$).

### 📘 Lecture notes — worked exercises

* **$3x^2 + 2\in O(x^2)$?** Yes: $3x^2+2\le 5x^2$ for $x\ge 1$. **$x^3\in O(x^2)$?** No: $x^3\le cx^2\iff x\le c$.
* **$3x + 4y + 5xy$** (both variables large): $\in O(xy)$ with $c = 12$, $x_0 = 1$; $\notin O(x)$ (fix $x$, let $y$
  grow: $5xy > cx$ once $y > c/5$); $\notin O(y)$ by symmetry. If $y$ is a fixed constant, it is $O(x)$.
* **$f(x) = x^2$ vs $g(x) = x^3$ ($x$ even) or $x$ ($x$ odd):** $f\notin O(g)$ (odd $x$), $f\notin\Omega(g)$ (even $x$),
  hence $f\notin o(g)$, $f\notin\omega(g)$, $f\notin\Theta(g)$ — the functions are incomparable.
* **Continuous vs discrete:** $dy/dx = \alpha y\Rightarrow y = e^{\alpha x}y(0)$; $y(x+1) = \beta y(x)\Rightarrow
  y = \beta^{x}y(0)$. Compare the bases $e^\alpha$ and $\beta$; with $\alpha\approx\beta$ (or $\beta = 1+\alpha$) the
  continuous model grows faster because $e^\alpha > 1 + \alpha$.

> 🎓 **Exam tip.** For piecewise or multi-variable functions, test *each* piece or direction separately: one bad
> subsequence of inputs is enough to break O or Ω.

---
## 8 · Asymptotics in bioinformatics

> 📍 **Where we are.** Back to the four analyses of the opening question. We now *measure* how the running time of real
> algorithms grows, read the exponent off a log–log plot, and use exact counts where measuring is impossible.

**Reading growth rates from data.** If a running time behaves like $T(n)\approx a\,n^k$, then
$\log T = \log a + k\log n$: on log–log axes the points lie on a straight line whose **slope is the exponent $k$**. We
estimate the slope from measured times by fitting a straight line (`np.polyfit` with degree 1 on the logarithms).

### 8.1 All-pairs comparison: $\binom n2\in\Theta(n^2)$

Comparing $n$ sequences with each other (to find duplicates, or to build a distance matrix for clustering or a
phylogeny) needs $\binom n2 = n(n-1)/2$ comparisons. We time it on SIMULATED sequences of length 100, counting mismatched
positions (the Hamming distance) between every pair.

In [ ]:
# ── 8.1 Time all-pairs Hamming distances among n SIMULATED sequences (length 100) ──
def all_pairs_mismatches(seqs: np.ndarray) -> int:
    """Total number of mismatched positions over all n(n−1)/2 pairs of rows (sequences)."""
    total = 0
    for i in range(len(seqs) - 1):
        total += int((seqs[i + 1:] != seqs[i]).sum())        # compare sequence i with all later sequences
    return total


rng_pairs = np.random.default_rng(SEED + 8)
sizes, times_pairs = [250, 500, 1000, 2000], []
for n in sizes:
    seqs = rng_pairs.integers(0, 4, size=(n, 100), dtype=np.int8)       # n random sequences over {0,1,2,3}
    runs = []
    for _ in range(3):                                                  # best of 3 runs reduces timing noise
        t0 = time.perf_counter()
        total = all_pairs_mismatches(seqs)
        runs.append(time.perf_counter() - t0)
    times_pairs.append(min(runs))
    print(f"n = {n:5d}: {n * (n - 1) // 2:>9,} pairs, mean mismatches per pair {total / (n * (n - 1) / 2):.1f} / 100, "
          f"time {times_pairs[-1]:.3f} s")
slope_pairs = np.polyfit(np.log(sizes), np.log(times_pairs), 1)[0]
print(f"log–log slope ≈ {slope_pairs:.2f}  (Θ(n²) predicts 2)")

🔍 **What just happened?** The number of pairs grows from 31,125 to 1,999,000 — a factor of 64 for an eight-fold larger
$n$, exactly $8^2$. Two random DNA sequences differ at about 75 of 100 positions (a random base matches with
probability 1/4), which is a sanity check of the simulation. The measured times are machine-dependent, so yours will
differ from ours (and change a little from run to run). Look at your own printed times: each doubling of $n$ should
multiply them by a factor between about 2 and 4, the factor approaching 4 as $n$ grows, and the fitted log–log slope
should lie somewhat below the predicted 2 (in our runs it came out between about 1.6 and 1.9). The reason is that for
small $n$ fixed overheads — such as starting each NumPy comparison — flatten the curve, which is why asymptotics only
speaks about large $n$. Extrapolating: comparing all pairs of 10 million reads would
need $5\times10^{13}$ comparisons — which is why duplicate detection sorts or hashes reads instead ($O(n\log n)$ or
$O(n)$).

### 8.2 Pairwise alignment: dynamic programming in $O(nm)$ instead of exponential enumeration

To align two DNA sequences $a$ (length $n$) and $b$ (length $m$), the **Needleman–Wunsch** algorithm fills an
$(n+1)\times(m+1)$ table $F$ in which $F[i][j]$ is the best score for aligning the first $i$ letters of $a$ with the
first $j$ letters of $b$:

$$F[i][j] = \max\big(F[i-1][j-1] + s(a_i, b_j),\ \ F[i-1][j] + \text{gap},\ \ F[i][j-1] + \text{gap}\big).$$

*In words:* the best alignment of the two prefixes ends either by pairing $a_i$ with $b_j$ (score +1 for a match, −1
for a mismatch here), or by aligning $a_i$ against a gap, or $b_j$ against a gap (−1 each); each cell needs three
look-ups, so the work is proportional to the number of cells, $(n+1)(m+1)\in\Theta(nm)$.

The naive alternative tries every alignment. The number of alignments of two sequences of length $n$ is the central
**Delannoy number** $D(n, n)$: 3, 13, 63, 321, … — it grows exponentially (roughly like $5.8^n$). The next cell checks
that dynamic programming finds the same best score as brute-force enumeration, and times both.

In [ ]:
# ── 8.2 Needleman–Wunsch (Θ(nm)) vs brute-force recursion over all alignments (exponential) ──
def needleman_wunsch(a: str, b: str, match: int = 1, mismatch: int = -1, gap: int = -1) -> int:
    """Best global alignment score of a and b by dynamic programming over an (n+1) × (m+1) table."""
    n, m = len(a), len(b)
    F = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        F[i][0] = i * gap
    for j in range(1, m + 1):
        F[0][j] = j * gap
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match if a[i - 1] == b[j - 1] else mismatch
            F[i][j] = max(F[i - 1][j - 1] + s, F[i - 1][j] + gap, F[i][j - 1] + gap)
    return F[n][m]


def brute_force_alignment(a: str, b: str, match: int = 1, mismatch: int = -1, gap: int = -1) -> int:
    """The same score by trying every alignment recursively (no table: exponentially many calls)."""
    if not a or not b:
        return gap * (len(a) + len(b))
    s = match if a[0] == b[0] else mismatch
    return max(brute_force_alignment(a[1:], b[1:]) + s, brute_force_alignment(a[1:], b) + gap,
               brute_force_alignment(a, b[1:]) + gap)


def delannoy(n: int, m: int) -> int:
    """Number of alignments (paths with steps →, ↓, ↘ through the table) of sequences of lengths n and m."""
    D = [[1] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i][j] = D[i - 1][j] + D[i][j - 1] + D[i - 1][j - 1]
    return D[n][m]


print("GATTACA vs GCATGCG: DP score", needleman_wunsch("GATTACA", "GCATGCG"),
      "| brute force", brute_force_alignment("GATTACA", "GCATGCG"))
print("number of alignments D(n, n), n = 1…10:", [delannoy(n, n) for n in range(1, 11)])

brute_sizes, brute_times = [5, 6, 7, 8, 9], []
for n in brute_sizes:
    a, b = simulate_dna(n, seed=1), simulate_dna(n, seed=2)
    t0 = time.perf_counter(); score_bf = brute_force_alignment(a, b); brute_times.append(time.perf_counter() - t0)
    assert score_bf == needleman_wunsch(a, b)
dp_sizes, dp_times = [100, 200, 400, 800], []
for n in dp_sizes:
    a, b = simulate_dna(n, seed=1), simulate_dna(n, seed=2)
    t0 = time.perf_counter(); needleman_wunsch(a, b); dp_times.append(time.perf_counter() - t0)
print("brute force times (n = 5…9):", [f"{t:.4f}" for t in brute_times], "s — ratio per +1 letter ≈",
      round(float(np.exp(np.polyfit(brute_sizes, np.log(brute_times), 1)[0])), 2))
print("DP times (n = 100…800):     ", [f"{t:.4f}" for t in dp_times], "s — log–log slope ≈",
      round(float(np.polyfit(np.log(dp_sizes), np.log(dp_times), 1)[0]), 2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].semilogy(brute_sizes, brute_times, "o-", color="tab:red")
axes[0].set_xlabel("sequence length n"); axes[0].set_ylabel("time (s, log scale)")
axes[0].set_title("brute force: a straight line on a log-y plot = exponential")
axes[1].loglog(dp_sizes, dp_times, "s-", color="tab:blue")
axes[1].set_xlabel("sequence length n = m (log)"); axes[1].set_ylabel("time (s, log)")
axes[1].set_title("dynamic programming: slope ≈ 2 on log–log = Θ(n²)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** For GATTACA vs GCATGCG both methods give the same best score, 0 (four matches, two
mismatches and two gaps: one common optimal alignment is G-ATTACA / GCATG-CG), and the `assert` confirmed agreement on all the
random sequences. The number of possible alignments explodes — 3, 13, 63, 321, … , 8,097,453 for $n = 10$ — and the
brute-force time grows by a roughly constant *factor* per extra letter (about 5–6 in our runs, in line with the number
of alignments, which grows by a factor approaching $3+2\sqrt2\approx 5.8$ per letter): a straight line on the log-y plot,
the signature of exponential growth. Dynamic programming handles 800 letters in a fraction of a second, and its log–log
slope is close to 2 (about 2.1 in our run; times are machine-dependent), as $\Theta(nm) = \Theta(n^2)$ predicts. Same answer, different growth rate: that is the power of a
good algorithm. (Real read aligners avoid even $\Theta(nm)$ against a whole genome by first locating k-mer seeds —
Module 04 — and aligning only near them.)

### 8.3 Exhaustive motif search: $O(4^k\cdot L)$ vs one pass over the sequence

Suppose we want the count of **every** k-mer in a sequence of length $L$. The brute-force way loops over all $4^k$
possible k-mers and scans the sequence for each: $O(4^k\cdot L)$ work. The clever way moves a window once along the
sequence and increments a counter for the k-mer it sees (`kmer_counts`): $O(L)$ steps (each handling a k-letter
string). For $k = 15$ and $L = 10^6$ the brute force would need $4^{15}\cdot10^6\approx10^{15}$ steps.

In [ ]:
# ── 8.3 Count all k-mers: brute force over Σ^k vs a single pass (SIMULATED 20-kb DNA) ──
dna_motif = simulate_dna(20_000, seed=33)
def count_occurrences(text: str, word: str) -> int:
    """Number of (possibly overlapping) occurrences of word in text: one scan of the text."""
    count, start = 0, text.find(word)
    while start != -1:
        count += 1
        start = text.find(word, start + 1)
    return count


for k in [4, 5, 6, 7]:
    t0 = time.perf_counter()
    brute = {"".join(word): count_occurrences(dna_motif, "".join(word))      # scan once per possible k-mer
             for word in itertools.product("ACGT", repeat=k)}
    t_brute = time.perf_counter() - t0
    t0 = time.perf_counter(); one_pass = kmer_counts(dna_motif, k); t_pass = time.perf_counter() - t0
    same = all(brute[w] == one_pass.get(w, 0) for w in brute)
    print(f"k = {k}: 4^k = {4 ** k:6d} k-mers | brute force {t_brute:7.3f} s | one pass {t_pass:.4f} s | "
          f"same counts? {same}")

🔍 **What just happened?** Both methods produce identical counts for every k-mer. The single pass takes a few
milliseconds for every $k$ (one sweep over 20,000 positions), whereas the brute-force time is driven by the $4^k$ factor:
it grows roughly four-fold for each step from $k = 4$ to $k = 7$ (compare the successive printed times), because each
increase of $k$ by one multiplies the number of k-mers to search for by 4. At $k = 7$ the one-pass method is already
tens of times faster, and the gap keeps multiplying by 4. (All times are machine-dependent; the *ratios* are what
matter.) The lesson is general: changing the algorithm
can move a problem from $O(4^k L)$ to $O(L)$, which no hardware upgrade can match.

### 8.4 How many phylogenetic trees? Super-exponential growth

A rooted binary phylogenetic tree on $n$ labelled species can be built by adding the species one at a time. With 2
species there is 1 tree; the 3rd species can be attached to any of the 3 branches (including a new branch above the
root); in general, a rooted binary tree on $k$ species has $2k - 1$ branches, so the $(k+1)$-st species has $2k - 1$
places to go. By the multiplication principle (Module 04),

$$\#\text{rooted binary trees on } n \text{ species} = 1\cdot 3\cdot 5\cdots(2n-3) = (2n-3)!! .$$

*In words:* the number of trees is the product of the first $n-1$ odd numbers ("double factorial") — and it grows
faster than any exponential $b^n$.

🧮 **Math refresher — factorials and double factorials.** $n! = 1\cdot 2\cdot 3\cdots n$ (e.g. $5! = 120$), and the
double factorial of an odd number multiplies only the odd numbers: $7!! = 1\cdot 3\cdot 5\cdot 7 = 105$. Both grow
faster than every exponential: $n!$ multiplies by $n+1$ at each step, while $b^n$ always multiplies by the same $b$.

In [ ]:
# ── 8.4 Enumerate rooted binary trees by inserting species one at a time; compare with (2n−3)!! ──
def insert_everywhere(tree, leaf: str) -> list:
    """All trees obtained by attaching `leaf` to one branch of `tree` (nested tuples), incl. above the root."""
    results = [(tree, leaf)]                                         # new branch above the current root
    if isinstance(tree, tuple):                                      # an internal node (left, right)
        left, right = tree
        results += [(t, right) for t in insert_everywhere(left, leaf)]
        results += [(left, t) for t in insert_everywhere(right, leaf)]
    return results


def all_rooted_trees(species: list) -> list:
    """Every rooted binary tree on the given species, built by successive insertion."""
    trees = [species[0]]
    for leaf in species[1:]:
        trees = [t for tree in trees for t in insert_everywhere(tree, leaf)]
    return trees


def double_factorial(m: int) -> int:
    """m!! = m·(m−2)·(m−4)···1 (1 for m ≤ 0)."""
    return math.prod(range(m, 0, -2)) if m > 0 else 1


print("trees on 3 species:", all_rooted_trees(["human", "chimp", "mouse"]))
for n in range(2, 8):
    trees = all_rooted_trees([f"s{i}" for i in range(1, n + 1)])
    print(f"n = {n}: enumerated {len(trees):6,} trees (all different: {len(set(map(str, trees))) == len(trees)}) "
          f"| (2n−3)!! = {double_factorial(2 * n - 3):,}")
for n in [10, 20, 50]:
    count = double_factorial(2 * n - 3)
    print(f"n = {n:2d}: (2n−3)!! ≈ {count:.3e} trees; at 10⁹ trees per second: {count / 1e9 / 3.15e7:.3g} years")

🔍 **What just happened?** For human, chimp and mouse there are exactly 3 rooted trees — ((human, chimp), mouse),
((human, mouse), chimp) and (human, (chimp, mouse)) — the three possible "who is closest to whom" answers. The insertion
procedure produces 1, 3, 15, 105, 945 and 10,395 distinct trees for 2 to 7 species, matching $(2n-3)!!$ exactly. Then
the numbers become absurd: 34 million trees for 10 species (still only 0.03 s at a billion trees per second),
$8.2\times10^{21}$ for 20 (260,000 years) and $2.8\times10^{76}$ for 50 ($8.7\times10^{59}$ years — the universe is
about $1.4\times10^{10}$ years old). Checking every tree for 50 bacterial strains is not slow — it is
impossible. This is why phylogenetics uses heuristic searches (rearranging a good tree locally), clustering algorithms
such as UPGMA (Module 06) and neighbour joining, or Bayesian MCMC sampling.

In [ ]:
# ── 8.5 Growth classes side by side: n², 2^n, n! and the number of rooted trees (2n−3)!! ──
n_vals = np.arange(2, 51)
log10_counts = {"n²": 2 * np.log10(n_vals), "2^n": n_vals * np.log10(2),
                "n!": np.array([math.lgamma(n + 1) / math.log(10) for n in n_vals]),
                "(2n−3)!!  rooted trees": np.array([math.log10(double_factorial(2 * n - 3)) for n in n_vals])}
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for name, values in log10_counts.items():
    ax.plot(n_vals, values, label=name, lw=2 if "trees" in name else 1.2)
ax.set_xlabel("number of species n"); ax.set_ylabel("log₁₀(count)"); ax.legend()
ax.set_title("the number of trees grows even faster than n!")
plt.tight_layout(); plt.show()
print("at n = 50: log10 of n² = %.1f, 2^n = %.1f, n! = %.1f, (2n−3)!! = %.1f"
      % tuple(v[-1] for v in log10_counts.values()))

🔍 **What just happened?** Plotting $\log_{10}$ of each count turns exponentials into straight lines ($2^n$ has slope
$\log_{10}2\approx 0.30$) and polynomials into slowly rising logarithmic curves ($n^2$ reaches only 3.4 at $n = 50$).
$n!$ and $(2n-3)!!$ bend *upwards* — super-exponential — and at $n = 50$ the tree count ($10^{76.4}$) exceeds even $50!$
($10^{64.5}$), and $2^{50}$ ($10^{15.1}$) by sixty orders of magnitude. In Big-O language: $2^n\in o(n!)$ and
$n!\in o\big((2n-3)!!\big)$.

### 8.6 Exponential vs logistic growth: "eventually" in biology

A single *Escherichia coli* cell dividing every 20 minutes (a commonly quoted doubling time in rich medium) would, if
nothing stopped it, produce $2^{72}$ cells in a day and $2^{144}$ in two days. Real populations follow exponential growth
only while resources last; the **logistic** model $N(t) = \dfrac{K}{1 + (K/N_0 - 1)e^{-rt}}$ levels off at the carrying
capacity $K$. Asymptotically the two belong to completely different classes: $e^{rt}\in\omega(t^k)$ for every $k$, while the
logistic curve is in $O(1)$ — bounded by a constant.

In [ ]:
# ── 8.6 What unchecked exponential growth would mean, and the logistic alternative (illustrative numbers) ──
cell_mass_g, earth_mass_kg = 1e-12, 5.97e24            # ≈ 1 picogram per E. coli cell; mass of the Earth
for hours in [24, 48]:
    doublings = hours * 3                                # one doubling every 20 min
    mass_kg = 2.0 ** doublings * cell_mass_g / 1000
    print(f"after {hours} h: 2^{doublings} ≈ {2.0 ** doublings:.2e} cells ≈ {mass_kg:.2e} kg = {mass_kg / earth_mass_kg:.3g} Earth masses")

r_per_h = 3 * math.log(2)                                # e^(r·t) doubles every 1/3 h
K, N0 = 1e9, 1.0                                         # carrying capacity (cells per mL, illustrative)
t = np.linspace(0, 16, 300)
exponential = N0 * np.exp(r_per_h * t)
logistic = K / (1 + (K / N0 - 1) * np.exp(-r_per_h * t))
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogy(t, exponential, label="exponential  N0·e^(rt)  ∈ ω(t^k)")
ax.semilogy(t, logistic, label="logistic, K = 10⁹ cells/mL  ∈ O(1)")
ax.axhline(K, color="0.5", ls=":")
ax.set_xlabel("time (h)"); ax.set_ylabel("cells per mL (log scale)"); ax.legend()
ax.set_title("identical at first, different asymptotic classes")
plt.tight_layout(); plt.show()
print(f"at t = 8 h: exponential {N0 * math.exp(r_per_h * 8):.3g}, logistic {K / (1 + (K / N0 - 1) * math.exp(-r_per_h * 8)):.3g}")

🔍 **What just happened?** After 24 hours of unchecked doubling a single bacterium would become $2^{72}\approx 4.7\times10^{21}$
cells, about 4,700 tonnes ($4.7\times10^{6}$ kg); after 48 hours, $2^{144}\approx 2.2\times10^{43}$ cells weighing about
$2.2\times10^{28}$ kg — roughly 3,700 times the mass of the Earth. Exponential growth cannot last, and in the flask it
does not: the logistic curve tracks the exponential for the first few hours (at 8 h the two are $1.68\times10^{7}$ and
$1.65\times10^{7}$), then bends and saturates at $K$. Asymptotics describes the long-run regime; the early regime of a
logistic curve is exponential. Module 14 derives both curves from differential equations.

### 🌉 Bridge — growth rates in computational biology and machine learning

* **Linear algebra costs.** Solving a linear system with $n$ unknowns by elimination costs $O(n^3)$ operations, while
  the cofactor (Laplace) expansion of a determinant costs $O(n!)$ — Module 08 shows both, and BiologyToML
  [Module 03](../../../../ml/tutorial/BiologyToML/notebooks/03_linear_algebra_for_data.ipynb) times `solve` against an explicit inverse.
* **Clustering.** Hierarchical clustering needs all $\binom n2$ pairwise distances — $\Theta(n^2)$ memory, at least
  $\Theta(n^2)$ time (BiologyToML [Module 12](../../../../ml/tutorial/BiologyToML/notebooks/12_clustering_kmeans_hierarchical.ipynb)): fine for 500 samples,
  impossible for a million cells, which is why single-cell pipelines cluster on nearest-neighbour graphs instead
  ([Module 30](../../../../ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb)).
* **Graphs.** Breadth-first search visits each node and edge once: $O(|V| + |E|)$ (Module 06).
* **Sequence models.** The forward algorithm of a hidden Markov model costs $O(L\cdot S^2)$ for a sequence of length
  $L$ and $S$ hidden states, instead of the $S^L$ paths it sums over ([Module 19](../../../../ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb)) — the
  same "dynamic programming beats enumeration" trick as Needleman–Wunsch.

✏️ **Your turn.** (a) Using the fitted slope from 8.2, predict how long your machine would need for a Needleman–Wunsch
alignment of two 10,000-base sequences, and of a 150-base read against a 3.1-Gb genome. (b) In 8.4, how many
*unrooted* binary trees are there on $n$ species? (Hint: it is $(2n-5)!!$, the same as the number of *rooted* trees on
$n-1$ species — verify it for $n = 4$: the three unrooted trees pair the species as $ab|cd$, $ac|bd$ or $ad|bc$, and
$(2\cdot 4-5)!! = 3$.)

## 📌 Key takeaways

- Asymptotics compares how functions $f, g: \mathbb R^{\ge0}\to\mathbb R^{\ge0}$ grow, **ignoring constant factors and
  small inputs**.
- $O(g) = \{h\mid\exists x_0\ge0, c>0: \forall x\ge x_0: h(x)\le c\,g(x)\}$ is a **set**; write $f\in O(g)$ (not
  $f = O(g)$). Prove membership with one pair of witnesses $(c, x_0)$; disprove it by defeating every pair.
- Rules: a sum behaves like its largest term; Big-O is transitive; bounds add for sums and multiply for products.
  Every polynomial of degree $k$ is in $O(x^k)$ (witnesses $\sum|a_i|$, 1).
- **log ≪ polynomial ≪ exponential ≪ factorial.** The base of a logarithm does not matter; the base of an exponential
  does: $b^x\in O(c^x)\iff b\le c$. "Eventually" can be astronomically late ($\ln x\le x^{0.1}$ only beyond $3.4\times10^{15}$).
- Ω (at least), Θ (same rate), o (strictly slower, ∀ c), ω (strictly faster, ∀ c); $f\in O(g)\iff g\in\Omega(f)$.
  Limits of ratios classify nice functions, but some pairs (the even/odd example) are incomparable.
- In bioinformatics: all-pairs comparison is $\Theta(n^2)$, Needleman–Wunsch $\Theta(nm)$ instead of exponentially many
  alignments, exhaustive motif search $O(4^kL)$ vs one $O(L)$ pass, and $(2n-3)!!$ phylogenetic trees make exhaustive tree
  search impossible beyond about a dozen species.
- Exponential growth ($\omega$ of every polynomial) never lasts in biology; logistic growth is $O(1)$.

---
## ✅ Checkpoint

1. Write the definition of $O(g)$ with quantifiers and explain the role of $c$ and of $x_0$ in one sentence each.
   <details><summary>Answer</summary>$O(g) = \{h\mid\exists x_0\ge 0, c>0: \forall x\ge x_0: h(x)\le c\,g(x)\}$. $c$ lets us ignore constant factors (machine speed, units); $x_0$ lets us ignore small inputs — only $x\ge x_0$ counts.</details>
2. Give two different pairs of witnesses for $4x\in O(x^2)$, and explain why $(c, x_0) = (1, 3)$ is not a pair of witnesses.
   <details><summary>Answer</summary>$4x\le cx^2\iff x\ge 4/c$, so e.g. $(1, 4)$ and $(2, 2)$ (any $c>0$ with $x_0 = 4/c$). With $(1, 3)$: at $x = 3$, $4x = 12 > 9 = x^2$, so the inequality fails at a point $\ge x_0$.</details>
3. Prove that $3x^2 + 2\in O(x^2)$ and that $x^3\notin O(x^2)$.
   <details><summary>Answer</summary>For $x\ge 1$: $2\le 2x^2$, so $3x^2+2\le 5x^2$ — witnesses $c = 5$, $x_0 = 1$. If $x^3\le cx^2$ for all $x\ge x_0$, then $x\le c$ for all $x\ge x_0$ (dividing by $x^2>0$), false at $x = \max(x_0, c)+1$.</details>
4. Prove transitivity: $f\in O(g)$ and $g\in O(h)$ imply $f\in O(h)$. Which witnesses do you use?
   <details><summary>Answer</summary>$f\le c_1g$ for $x\ge x_1$, $g\le c_2h$ for $x\ge x_2$; for $x\ge\max(x_1,x_2)$, $f\le c_1c_2h$. Witnesses $c = c_1c_2$, $x_0 = \max(x_1, x_2)$.</details>
5. Rank from slowest to fastest growth: $2^x$, $x\log x$, $\log_{10}x$, $x^3$, $\ln x$, $x!$, $\sqrt x$, $3^x$.
   <details><summary>Answer</summary>$\log_{10}x$ and $\ln x$ (same class, Θ of each other) < $\sqrt x$ < $x\log x$ < $x^3$ < $2^x$ < $3^x$ < $x!$.</details>
6. True or false: (a) $\log_2 x\in\Theta(\ln x)$; (b) $4^x\in O(2^x)$; (c) $x^{100}\in O(1.01^x)$; (d) $\ln x\in O(x^{0.01})$.
   <details><summary>Answer</summary>(a) True: $\log_2x = \ln x/\ln 2$. (b) False: $4^x/2^x = 2^x\to\infty$ ($b\le c$ fails). (c) True: every polynomial is in $O(b^x)$ for $b>1$ (the crossing is very late). (d) True: $\log x\in O(x^\epsilon)$ for every $\epsilon>0$.</details>
7. Prove $f\in O(g)\iff g\in\Omega(f)$.
   <details><summary>Answer</summary>$f(x)\le c\,g(x)\iff g(x)\ge\frac1c f(x)$. Keep the same $x_0$; witnesses $d = 1/c$ for Ω, and conversely $c = 1/d$.</details>
8. Is $3x + 4y + 5xy$ in $O(x)$? In $O(y)$? In $O(xy)$? What changes if $y$ is a fixed constant?
   <details><summary>Answer</summary>Not $O(x)$ (fix $x$, let $y\to\infty$: $5xy > cx$ once $y > c/5$), not $O(y)$ (symmetric), but in $O(xy)$: $\le 12xy$ for $x, y\ge 1$. If $y$ is constant, $(3+5y)x + 4y\in O(x)$.</details>
9. For $f(x) = x^2$ and $g(x) = x^3$ (even $x$), $x$ (odd $x$), decide $f\in O(g)$, $\Omega(g)$, $o(g)$, $\omega(g)$.
   <details><summary>Answer</summary>None. Odd $x$: $x^2\le cx$ fails for $x>c$ (not O, hence not o). Even $x$: $x^2\ge dx^3$ fails for $x>1/d$ (not Ω, hence not ω).</details>
10. Explain why "$o(g) = O(g)$ but not $\Omega(g)$" can fail, using $h(x) = x^2$ (even $x$), $x$ (odd $x$) and $g(x) = x^2$.
    <details><summary>Answer</summary>$h\le x^2$ (in O), odd $x$ give $h = x < d x^2$ (not in Ω), but with $c = 1/2$ every even $x$ has $h = x^2\not< \frac12x^2$, so $h\notin o(x^2)$. The shortcut only holds when $\lim h/g$ exists.</details>
11. If $\frac{dy}{dx} = \alpha y$ and $y(x+1) = \beta y(x)$ with $\alpha = \beta = 0.3$, which grows faster? What is the solution of the discrete model?
    <details><summary>Answer</summary>$y = e^{0.3x}y(0)$ vs $y = 0.3^x y(0)$; compare the bases $e^{0.3}\approx1.35$ and 0.3 — the continuous one grows, the discrete one even decays. (With $\beta = 1.3$: $1.3 < e^{0.3}$, continuous still wins.) Discrete solution: $\beta^xy(0)$.</details>
12. How many rooted binary trees exist on 6 species? Roughly how many alignments of two 10-letter sequences? Why do these numbers matter?
    <details><summary>Answer</summary>$9!! = 1\cdot3\cdot5\cdot7\cdot9 = 945$ trees; $D(10,10) = 8{,}097{,}453$ alignments. Both grow (super-)exponentially, so exhaustive enumeration is hopeless for realistic sizes; dynamic programming (alignment, $\Theta(nm)$) or heuristics (trees) are needed.</details>

## 🧠 Questions & discussion

1. Big-O hides constants. Give an example in which an $O(n^2)$ method is the better practical choice than an $O(n\log n)$ method. When does the asymptotic advice start to matter?
2. A single-cell atlas grows from $10^4$ to $10^7$ cells. Which steps of a typical analysis (normalisation, PCA, building a nearest-neighbour graph, all-pairs distances, clustering) become bottlenecks, and why?
3. Why do algorithm designers usually analyse the *worst case*? Relate your answer to the even/odd example of Section 7.2. When would the average case be more relevant for a biologist?
4. The number of phylogenetic trees grows super-exponentially, yet phylogeneticists routinely infer trees for thousands of species. What does this tell you about the difference between "the search space is huge" and "the problem cannot be solved well"?
5. Exponential growth "eventually" beats any polynomial, and logistic growth is eventually constant. In an epidemic, which regime matters for policy decisions in the first two weeks, and which over a year?

## 🚀 Bonus — going further

- **Stirling's formula.** Show numerically that $n!\approx\sqrt{2\pi n}\,(n/e)^n$ (the ratio tends to 1) and use it to prove $\log(n!)\in\Theta(n\log n)$ — the reason sorting by comparisons needs $\Omega(n\log n)$ comparisons.
- **Master theorem preview.** Merge sort splits a list in half and merges in linear time: $T(n) = 2T(n/2) + n$. Implement it, time it for $n = 10^4\ldots10^6$ and check that $T(n)/(n\log_2 n)$ levels off.
- **Hirschberg's trick.** Needleman–Wunsch needs $\Theta(nm)$ *memory* as well as time. Read about Hirschberg's divide-and-conquer algorithm, which keeps $\Theta(nm)$ time but uses only linear memory.
- **Counting unrooted trees.** Prove that there are $(2n-5)!!$ unrooted binary trees on $n\ge 3$ species by the insertion argument (an unrooted tree on $k$ leaves has $2k-3$ edges). How many are there for 10 species?
- **Further reading.** Liben-Nowell, *Connecting Discrete Mathematics and Computer Science*, §6.2 (asymptotics); Cormen, Leiserson, Rivest & Stein, *Introduction to Algorithms*, Ch. 3 (growth of functions).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            05 · Asymptotics: how fast functions grow
Input:             the "zoo" of growth functions; SIMULATED random sequences (NumPy, simulate_dna)
Output:            witness pairs (c, x0), counterexamples, limits of ratios, crossing points,
                   measured running times and log–log slopes, counts of alignments and trees
Method:            quantifier definitions of O, Ω, Θ, o, ω; SymPy limits; brentq for crossings;
                   time.perf_counter + np.polyfit on log–log data; dynamic programming vs enumeration
Key equation:      O(g) = {h | ∃x0 ≥ 0, c > 0 : ∀x ≥ x0 : h(x) ≤ c·g(x)};  b^x ∈ O(c^x) ⇔ b ≤ c
Key parameters:    constants c and thresholds x0; input sizes n, m, k, L; bases of logs and exponentials
What changed when I modified …  (e.g. the all-pairs constant in 3.1; the operation budget in 5.2)
One failure mode:  trusting a finite check (a grid, a plot) as a proof of "for all x ≥ x0";
                   or a limit test when the limit of f/g does not exist
Connection to 02-680: Big-O as a set; properties; polynomials; log ≪ poly ≪ exp; Ω, Θ, o, ω; worked exercises
```

## 📚 References

- Liben-Nowell D. (2022) *Connecting Discrete Mathematics and Computer Science*, 2nd ed. Cambridge University Press — §6.2 (asymptotics: O, Ω, Θ, o, ω, and their properties).
- Cormen T. H., Leiserson C. E., Rivest R. L., Stein C. (2022) *Introduction to Algorithms*, 4th ed. MIT Press — Ch. 3 (characterising running times) and the chapter on dynamic programming.
- Betti M. *Mathematics and Statistics for the Quantitative Sciences* — limits, exponential and logistic growth.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Needleman S. B., Wunsch C. D. (1970) A general method applicable to the search for similarities in the amino acid sequence of two proteins. *Journal of Molecular Biology* 48:443–453.
- Felsenstein J. (2004) *Inferring Phylogenies*. Sinauer — Ch. 3 (counting trees: $(2n-3)!!$ rooted and $(2n-5)!!$ unrooted binary trees).
- Durbin R., Eddy S., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis*. Cambridge University Press — Ch. 2 (pairwise alignment by dynamic programming).

## ➡️ Next

**[Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb)** — the last foundation module turns relations into **graphs**: gene-regulatory and protein-interaction
networks, metabolic networks and phylogenies. You will use this module constantly there: breadth-first search runs in
$O(|V| + |E|)$, a complete graph has $\binom n2\in\Theta(n^2)$ edges, and the count of rooted binary trees you just met,
$(2n-3)!!$, is why phylogenetic trees are built by clever algorithms such as UPGMA rather than by enumeration.